# Lung case study: lung cancer *vs* pulmonary tuberculosis (MetaboLights MTBLS6990)

This notebook recomputes everything in the article that concerns the lung cohort, from the deposited
feature tables (`data/lung_{10,2,0.5}mz_2min.pkl`). It covers the first configuration (all 2 375 bins
of the 2 m/z × 2 min grid, per-sample-maximum normalization; Tables 3 and 5) and its controls
(whole-protocol permutation, run-order check, random descriptors, learning curve, three-class
demonstration; Tables S15, S26 and S27), the bins it selects (Figure S1C–D), and the second configuration
(label-free stability filter and probabilistic quotient normalization; Table 4, Figure 3, Tables S15–S19,
Figure S5). Section 7 reads the raw files when they are available (`MINUT_RAW/lung*/*.mzML`, 236
injections) for the polypropylene-glycol series, the ions of the old Table 3, the phenylalanylphenylalanine region and the
identity of the stable bins (Tables S11–S14, S16 and S17); otherwise it reuses the per-sample tables of
the last raw pass.

In [1]:
import json, os, platform, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, sklearn
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import mannwhitneyu, rankdata
from sklearn.cluster import DBSCAN
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, cross_val_score, permutation_test_score
from sklearn.pipeline import Pipeline
from sklearn.utils import resample

sys.path.insert(0, str(Path.cwd()))
import minut as mn
from minut import GRIDS, H, K, Na, NH4, H2O, D13C, PPG_END, PPG_UNIT, match_ppg, peak, read_scans, bin_maximum

warnings.filterwarnings("ignore")
GRID = GRIDS["lung"]; NB, NMZ = 2375, 475
OUT = mn.RESULTS / "lung"; FIG = OUT / "figures"; TEX = OUT / "tex"; FIG.mkdir(parents=True, exist_ok=True); TEX.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 70)
NUM = {}
print(f"Python {platform.python_version()}, numpy {np.__version__}, scipy {scipy.__version__}, scikit-learn {sklearn.__version__}")
print("raw lung files:", mn.raw_available("lung*"), "| MAF:", mn.raw_available("maf"))

Python 3.11.9, numpy 1.26.4, scipy 1.16.3, scikit-learn 1.3.0
raw lung files: True | MAF: True


## 1. Data

236 clinical sera (143 lung cancer, 93 tuberculosis) with the discovery (85) and validation (151) cohorts
predefined by the source study, acquired on two days. The first configuration uses the intensity and m/z
blocks of the 2 m/z × 2 min grid, 4 750 descriptors in all.

In [2]:
X, y, train, test, runs = mn.load_lung()
I, M, R = mn.decoded_blocks(np.stack(runs.output_vector.to_list()), GRID)         # intensity, m/z, RT of the bin maxima
runs["date"] = runs.sample_name.str.extract(r"lcy(\d{8})_")[0]; runs["run_index"] = runs.sample_name.str.extract(r"_(\d+)$")[0].astype(int)
disc = np.zeros(len(y), bool); disc[train] = True; lc = y == 1
dates = pd.crosstab([runs.cohort, runs.date], runs.disease); display(dates); dates.to_csv(OUT / "lung_permutation_dates.csv")
print(f"{len(y)} sera: {int(lc.sum())} lung cancer, {int((~lc).sum())} tuberculosis; discovery {len(train)} ({int(y[train].sum())} LC), validation {len(test)} ({int(y[test].sum())} LC); {X.shape[1]} descriptors")
NUM.update(n=len(y), n_lc=int(lc.sum()), n_tb=int((~lc).sum()), n_disc=len(train), n_val=len(test), n_val_lc=int(y[test].sum()), n_val_tb=int((y[test] == 0).sum()), disc_date=sorted(runs.date[disc].unique()), val_date=sorted(runs.date[~disc].unique()))

,disease,Pulmonary Tuberculosis,lung cancer
cohort,date,,
discovery,20201008,30,55
validation,20201125,63,88


236 sera: 143 lung cancer, 93 tuberculosis; discovery 85 (55 LC), validation 151 (88 LC); 4750 descriptors


## 2. First configuration (Tables 3 and 5, MINUT rows)

Ten-fold cross-validation on the discovery cohort, with the six most important descriptors re-selected
inside each fold (selector forest seed 2024, classifier seed 42, which is also the seed of the final
model). The final model keeps the descriptors whose importance reaches the seventh largest value, hence
seven descriptors, and is scored on the 151 validation sera. The permutation p-value is the one of the
original analysis: 100 label permutations of the test set, five folds, on the selected descriptors.

In [3]:
cv = cross_val_score(Pipeline([("select", mn.GiniSelector(6, 2024)), ("forest", RandomForestClassifier(n_estimators=100, random_state=42))]), X[train], y[train], cv=10)
keep = mn.threshold_selection(X[train], y[train], 6, random_state=2024)
model, pred, proba, sc = mn.holdout(X, y, train, test, keep, seed=42)
_, _, perm_p = permutation_test_score(model, X[test][:, keep], y[test], cv=5, n_permutations=100, scoring="accuracy", random_state=42)
desc = lambda i: f"{'I' if i < NB else 'mz'}[bin {i % NB}]"
sel1 = [int(i) for i in np.flatnonzero(keep)]
print(f"10-fold CV accuracy {cv.mean():.4f} ± {cv.std():.4f} (selector seed 2024, classifier seed 42)")
print(f"validation: accuracy {sc['test_accuracy']:.4f}, AUC {sc['test_auc']:.4f}, bootstrap {sc['bootstrap_accuracy']:.3f} ± {sc['bootstrap_sd']:.3f}, permutation p {perm_p:.4f}")
print("selected descriptors:", [f"{desc(i)} ({mn.bin_label(i % NB, GRID)})" for i in sel1])
cw = mn.classification_summary(y[test], pred, proba, positive="cancer", negative="tuberculosis")
pd.DataFrame([dict(cv_accuracy=cv.mean(), cv_sd=cv.std(), permutation_p=perm_p, **sc)]).to_csv(OUT / "lung.csv", index=False, float_format="%.6f")
NUM.update(stage1_cv=cv.mean(), stage1_cv_sd=cv.std(), stage1_acc=sc["test_accuracy"], stage1_auc=sc["test_auc"], stage1_desc=[f"{'I' if c < NB else 'mz'}:{c % NB}" for c in sel1])

10-fold CV accuracy 1.0000 ± 0.0000 (selector seed 2024, classifier seed 42)
validation: accuracy 1.0000, AUC 1.0000, bootstrap 1.000 ± 0.000, permutation p 0.0099
selected descriptors: ['I[bin 25] (m/z [100,102) RT [2,4) min)', 'I[bin 545] (m/z [190,192) RT [4,6) min)', 'I[bin 813] (m/z [726,728) RT [4,6) min)', 'I[bin 818] (m/z [736,738) RT [4,6) min)', 'I[bin 874] (m/z [848,850) RT [4,6) min)', 'I[bin 1409] (m/z [968,970) RT [6,8) min)', 'mz[bin 256] (m/z [562,564) RT [2,4) min)']


The first-submission notebook (`Archive/CaseStudy/lung/LungCancer.ipynb`) prints 0.989 ± 0.033 for this cross-validation.
Two of its settings differ from the protocol above: the classifier seed of its CV pipeline is 2024 (the
final model uses 42), and its samples follow the order of the MetaboLights metadata file rather than that
of `data/lung_injections.csv`, which changes the composition of the ten unshuffled folds. The cell below
reproduces that ordering. With it and seed 2024 we get 0.989 ± 0.033 again; with seed 42, the one of the
article, the CV is 1.00 ± 0.00 in both orderings.

In [4]:
s_ = pd.read_csv(mn.DATA / "metadata" / "s_MTBLS6990.txt", sep="\t"); a_ = pd.read_csv(mn.DATA / "metadata" / "a_MTBLS6990_LC-MS_untargeted_positive_reverse-phase_metabolite_profiling.txt", sep="\t")
isa = s_.merge(a_, on="Sample Name", how="inner"); isa = isa[isa["Characteristics[Organism]"] != "blank"][["Factor Value[Cohort]", "Factor Value[Disease]", "Raw Spectral Data File"]]
isa["run"] = isa["Raw Spectral Data File"].astype(str).str.split("/").str[-1].str.split(".").str[0]
pk = pd.read_pickle(mn.DATA / "lung_2mz_2min.pkl"); pk["run"] = pk.echantillon.str.split("/").str[-1].str.split(".").str[0]
nbdf = isa.merge(pk, on="run", how="inner").dropna(); nbdf = nbdf[nbdf["Factor Value[Disease]"] != "Healthy Control"]
nb_train = nbdf[nbdf["Factor Value[Cohort]"] != "validation"]; Xnb = np.stack(nb_train.output_vector.to_list())[:, :2 * NB]; ynb = (nb_train["Factor Value[Disease]"] == "lung cancer").to_numpy().astype(int)
assert set(nb_train.run) == set(runs.sample_name[train])
cv_isa = {s: cross_val_score(Pipeline([("select", mn.GiniSelector(6, 2024)), ("forest", RandomForestClassifier(n_estimators=100, random_state=s))]), Xnb, ynb, cv=10) for s in (2024, 42)}
print(f"metadata ordering: classifier seed 2024 -> {cv_isa[2024].mean():.4f} ± {cv_isa[2024].std():.4f} (deposited notebook); seed 42 -> {cv_isa[42].mean():.4f} ± {cv_isa[42].std():.4f} (article)")

metadata ordering: classifier seed 2024 -> 0.9889 ± 0.0333 (deposited notebook); seed 42 -> 1.0000 ± 0.0000 (article)


## 3. Bin size and number of descriptors (Figure S1, panels C and D)

Both sweeps use the discovery cohort only: five-fold cross-validation with the selection refitted in each
fold, and the seeds of the original notebook (selector 2024; classifier 1791 for the bin-size sweep with
50 descriptors, 1787 for the descriptor sweep). As above, the unshuffled folds depend on the sample order,
and the original notebook used the metadata order, so we run the sweeps in that order and, for
comparison, in the order of `data/lung_injections.csv`.

In [5]:
def isa_train(width):
    pk_ = pd.read_pickle(mn.DATA / f"lung_{width:g}mz_2min.pkl"); pk_["run"] = pk_.echantillon.str.split("/").str[-1].str.split(".").str[0]
    d_ = isa.merge(pk_, on="run", how="inner").dropna(); d_ = d_[(d_["Factor Value[Disease]"] != "Healthy Control") & (d_["Factor Value[Cohort]"] != "validation")]
    V_ = np.stack(d_.output_vector.to_list()); nb_ = V_.shape[1] // 3
    return V_[:, :2 * nb_], (d_["Factor Value[Disease]"] == "lung cancer").to_numpy().astype(int)
def sweep_cv(Xw, yw, k, seed):
    return cross_val_score(Pipeline([("select", mn.GiniSelector(k, 2024)), ("forest", RandomForestClassifier(n_estimators=100, random_state=seed))]), Xw, yw, cv=5)
rows = []
for width in (10, 2, 0.5):
    Xi, yi = isa_train(width); Xw, yw, trw, _, _ = mn.load_lung(mz_width=width)
    s_isa, s_dep = sweep_cv(Xi, yi, 50, 1791), sweep_cv(Xw[trw], yw[trw], 50, 1791)
    rows.append(dict(mz_width=width, n_bins=Xw.shape[1] // 2, cv5_mean=s_isa.mean(), cv5_sd=s_isa.std(), cv5_mean_table_order=s_dep.mean(), cv5_sd_table_order=s_dep.std()))
    print(f"{width:>4g} m/z: {Xw.shape[1] // 2:>5d} bins, CV accuracy {s_isa.mean():.3f} ± {s_isa.std():.3f} (metadata order, as the figure) / {s_dep.mean():.3f} ± {s_dep.std():.3f} (table order)")
binsize = pd.DataFrame(rows); binsize.to_csv(OUT / "figS1C_bin_size.csv", index=False, float_format="%.6f")
fig, ax = plt.subplots(figsize=(6, 4)); ax.errorbar(binsize.mz_width, binsize.cv5_mean, yerr=binsize.cv5_sd, fmt="-o", capsize=5, color="tab:blue", ecolor="gray", label="Mean CV accuracy ± 1 STD")
ax.set_xlabel("M/z bin size"); ax.set_ylabel("Mean cross-validation accuracy on training set"); ax.grid(True, ls="--", alpha=.5); ax.legend(loc="lower right")
plt.tight_layout(); fig.savefig(FIG / "figS1C_bin_size.pdf"); fig.savefig(FIG / "figS1C_bin_size.png", dpi=150); plt.close(fig)
Xi, yi = isa_train(2); sizes = list(range(1, 50)); res_isa = [sweep_cv(Xi, yi, k, 1787) for k in sizes]; res_dep = [sweep_cv(X[train], y[train], k, 1787) for k in sizes]
sweep = pd.DataFrame(dict(n_features=sizes, cv5_mean=[r.mean() for r in res_isa], cv5_sd=[r.std() for r in res_isa], cv5_mean_table_order=[r.mean() for r in res_dep], cv5_sd_table_order=[r.std() for r in res_dep]))
sweep.to_csv(OUT / "figS1D_n_features_sweep.csv", index=False, float_format="%.6f")
print("descriptor sweep (metadata order): " + ", ".join(f"{k}: {m:.3f}" for k, m in zip(sizes[:6], sweep.cv5_mean[:6])) + ", then " + ("1.000 throughout" if (sweep.cv5_mean[6:] == 1).all() else "see the table"))
fig, ax = plt.subplots(figsize=(7, 4)); ax.errorbar(sizes, sweep.cv5_mean, yerr=sweep.cv5_sd, fmt="-o", capsize=4, ms=4, color="tab:blue", ecolor="gray", label="Mean CV Accuracy ± 1 STD")
ax.set_xlabel("Number of Selected Features"); ax.set_ylabel("Mean Cross-Validation Accuracy"); ax.grid(True, ls="--", alpha=.5); ax.legend(loc="lower right")
plt.tight_layout(); fig.savefig(FIG / "figS1D_n_features.pdf"); fig.savefig(FIG / "figS1D_n_features.png", dpi=150); plt.close(fig); print("Figure S1C-D written")

  10 m/z:   475 bins, CV accuracy 1.000 ± 0.000 (metadata order, as the figure) / 1.000 ± 0.000 (table order)


   2 m/z:  2375 bins, CV accuracy 1.000 ± 0.000 (metadata order, as the figure) / 1.000 ± 0.000 (table order)


 0.5 m/z:  9500 bins, CV accuracy 1.000 ± 0.000 (metadata order, as the figure) / 0.988 ± 0.024 (table order)


descriptor sweep (metadata order): 1: 0.965, 2: 0.965, 3: 1.000, 4: 1.000, 5: 1.000, 6: 1.000, then 1.000 throughout
Figure S1C-D written


The deposited panel D of Figure S1 shows 0.976 and 0.988 for one and two descriptors and 1.00 from three onward. Our
recomputation gives 0.965 for one and two descriptors and 1.00 from three onward in the metadata order
(0.965, 0.976, 0.988, then 1.00 with two dips at 0.988 in the table order). The original notebook has no
stored output for this figure, so we cannot tell where its first two points came from; the plateau is
reproduced, the first two points are not exactly.

## 4. Controls of the first configuration (Tables S15, S26 and S27)

* Whole-protocol permutation test: we permute the discovery labels, rerun selection and fitting, and score
  the resulting model on the untouched validation cohort (500 permutations; 100 for the ten-fold CV
  score). p = (1 + #{null ≥ observed}) / (N + 1).
* Run-order check: acquisition date, run index within the day and global run order as the only predictors.
* Random-descriptor control: seven descriptors drawn at random instead of selected (500 draws).
* Learning curve: stratified subsamples of the discovery cohort (20 repeats per size), scored on the 151
  validation sera.
* Three classes: the 100 healthy-control runs of MTBLS6990 are added (discovery 37/30/55, validation
  63/63/88), same representation, selector and forest, k = 7 and k = 50, plus the three pairwise problems.

In [6]:
def fit_and_score1(Xk, yk, tr, te, seed_select=2024, seed_forest=42, k=6):
    kp = mn.threshold_selection(Xk[tr], yk[tr], k, random_state=seed_select)
    m = RandomForestClassifier(n_estimators=100, random_state=seed_forest).fit(Xk[tr][:, kp], yk[tr])
    return accuracy_score(yk[te], m.predict(Xk[te][:, kp])), roc_auc_score(yk[te], m.predict_proba(Xk[te][:, kp])[:, 1])

# the same random stream (seed 2024) feeds, in this order, the 500 permutations, the 100 CV permutations and the 500 random draws
t0 = time.time(); rng = np.random.RandomState(2024); perm_rows = []
for b in range(500):
    yp = y.copy(); yp[train] = rng.permutation(y[train])            # validation labels untouched
    a, u = fit_and_score1(X, yp, train, test); perm_rows.append(dict(perm=b, validation_accuracy=a, validation_auc=u))
perm = pd.DataFrame(perm_rows)
p_acc = (1 + (perm.validation_accuracy >= sc["test_accuracy"]).sum()) / 501; p_auc = (1 + (perm.validation_auc >= sc["test_auc"]).sum()) / 501
cv_null = np.array([cross_val_score(Pipeline([("select", mn.GiniSelector(6, 2024)), ("forest", RandomForestClassifier(n_estimators=100, random_state=42))]), X[train], rng.permutation(y[train]), cv=10).mean() for _ in range(100)])
p_cv = (1 + (cv_null >= cv.mean()).sum()) / 101
baseline = max(y[test].mean(), 1 - y[test].mean())
print(f"{time.time() - t0:.0f} s. whole-protocol null: validation accuracy {perm.validation_accuracy.mean():.3f} ± {perm.validation_accuracy.std(ddof=0):.3f} (max {perm.validation_accuracy.max():.3f}; majority baseline {baseline:.3f}), "
      f"AUC {perm.validation_auc.mean():.3f} (max {perm.validation_auc.max():.3f}); p(accuracy) = {p_acc:.4f}, p(AUC) = {p_auc:.4f}; training-CV null {cv_null.mean():.3f} (max {cv_null.max():.3f}), p = {p_cv:.4f}")
# run-order check
order = runs.sort_values(["date", "run_index"]).index; g = np.empty(len(runs), int); g[order] = np.arange(len(runs))
Mo = np.column_stack([pd.to_datetime(runs.date).map(pd.Timestamp.toordinal).to_numpy(), runs.run_index.to_numpy(), g])
fo = RandomForestClassifier(n_estimators=100, random_state=42).fit(Mo[train], y[train])
ro = dict(validation_accuracy=accuracy_score(y[test], fo.predict(Mo[test])), validation_auc=roc_auc_score(y[test], fo.predict_proba(Mo[test])[:, 1]),
          cv_accuracy=cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), Mo[train], y[train], cv=10).mean(), majority_baseline=baseline)
print(f"run-order check: validation accuracy {ro['validation_accuracy']:.3f} (baseline {baseline:.3f}), AUC {ro['validation_auc']:.3f}; discovery CV {ro['cv_accuracy']:.3f}")
# random-descriptor control
rand = np.array([accuracy_score(y[test], RandomForestClassifier(n_estimators=100, random_state=42).fit(X[train][:, kp], y[train]).predict(X[test][:, kp])) for kp in (rng.choice(X.shape[1], 7, replace=False) for _ in range(500))])
print(f"seven random descriptors (500 draws): validation accuracy {rand.mean():.3f} ± {rand.std():.3f} (max {rand.max():.3f})")
perm.to_csv(OUT / "lung_permutation_null.csv", index=False, float_format="%.6f"); pd.DataFrame(dict(cv_null_accuracy=cv_null)).to_csv(OUT / "lung_permutation_cv_null.csv", index=False, float_format="%.6f")
pd.DataFrame(dict(random7_validation_accuracy=rand)).to_csv(OUT / "lung_random_descriptor_control.csv", index=False, float_format="%.6f")
pd.DataFrame([dict(observed_validation_accuracy=sc["test_accuracy"], observed_validation_auc=sc["test_auc"], observed_cv_accuracy=cv.mean(), observed_cv_sd=cv.std(), selected_descriptors=";".join(desc(i) for i in sel1), n_perm=500,
                   null_accuracy_mean=perm.validation_accuracy.mean(), null_accuracy_max=perm.validation_accuracy.max(), null_auc_mean=perm.validation_auc.mean(), null_auc_max=perm.validation_auc.max(), p_accuracy=p_acc, p_auc=p_auc,
                   n_perm_cv=100, cv_null_mean=cv_null.mean(), cv_null_max=cv_null.max(), p_cv=p_cv, run_order_validation_accuracy=ro["validation_accuracy"], run_order_validation_auc=ro["validation_auc"], run_order_cv_accuracy=ro["cv_accuracy"],
                   validation_majority_baseline=baseline, random7_mean=rand.mean(), random7_max=rand.max())]).to_csv(OUT / "lung_permutation_summary.csv", index=False, float_format="%.6f")

1044 s. whole-protocol null: validation accuracy 0.522 ± 0.089 (max 0.795; majority baseline 0.583), AUC 0.517 (max 0.885); p(accuracy) = 0.0020, p(AUC) = 0.0020; training-CV null 0.561 (max 0.693), p = 0.0099


run-order check: validation accuracy 0.583 (baseline 0.583), AUC 0.561; discovery CV 0.351


seven random descriptors (500 draws): validation accuracy 0.661 ± 0.157 (max 1.000)


In [7]:
rows = []
for n in (10, 15, 20, 30, 40, 50, 60, 70, 85):
    for r in range(20):
        sub = train if n >= len(train) else train[next(StratifiedShuffleSplit(1, train_size=n, random_state=1000 * n + r).split(X[train], y[train]))[0]]
        kp = mn.threshold_selection(X[sub], y[sub], 6, random_state=2024); m = RandomForestClassifier(n_estimators=100, random_state=42).fit(X[sub][:, kp], y[sub]); p = m.predict(X[test][:, kp])
        rows.append(dict(dataset="lung", n_train=len(sub), n_train_pos=int(y[sub].sum()), repeat=r, accuracy=accuracy_score(y[test], p), balanced_accuracy=balanced_accuracy_score(y[test], p), auc=roc_auc_score(y[test], m.predict_proba(X[test][:, kp])[:, 1])))
        if n >= len(train):
            break
lc_rep = pd.DataFrame(rows)
lcurve = lc_rep.groupby("n_train").agg(n_train_pos=("n_train_pos", "first"), repeats=("repeat", "count"), accuracy_mean=("accuracy", "mean"), accuracy_sd=("accuracy", "std"), accuracy_min=("accuracy", "min"),
                                       balanced_mean=("balanced_accuracy", "mean"), auc_mean=("auc", "mean"), auc_sd=("auc", "std")).reset_index(); lcurve.insert(0, "dataset", "lung")
display(lcurve.round(3)); lc_rep.to_csv(OUT / "learning_curve_repeats.csv", index=False, float_format="%.6f"); lcurve.to_csv(OUT / "learning_curve_summary.csv", index=False, float_format="%.6f")

,dataset,n_train,n_train_pos,repeats,accuracy_mean,accuracy_sd,accuracy_min,balanced_mean,auc_mean,auc_sd
0,lung,10,6,20,0.834,0.135,0.470,0.858,0.998,0.004
1,lung,15,10,20,0.843,0.152,0.437,0.866,0.973,0.072
2,lung,20,13,20,0.925,0.082,0.709,0.932,0.998,0.004
3,lung,30,19,20,0.939,0.083,0.748,0.948,1.000,0.000
4,lung,40,26,20,0.985,0.030,0.887,0.987,1.000,0.000
5,lung,50,32,20,0.992,0.019,0.927,0.993,1.000,0.000
6,lung,60,39,20,1.000,0.000,1.000,1.000,1.000,0.000
7,lung,70,45,20,0.999,0.004,0.980,0.999,1.000,0.000
8,lung,85,55,1,1.000,NaN,1.000,1.000,1.000,NaN


In [8]:
meta_all = mn.lung_table(); CLASSES = {"Healthy Control": 0, "Pulmonary Tuberculosis": 1, "lung cancer": 2}; NAMES = ["healthy", "TB", "cancer"]
clin = meta_all.disease.isin(CLASSES).to_numpy(); X3 = mn.lung_blocks(meta_all)[clin]; df3 = meta_all[clin].reset_index(drop=True); y3 = df3.disease.map(CLASSES).to_numpy()
tr3 = np.flatnonzero(df3.cohort == "discovery"); te3 = np.flatnonzero(df3.cohort == "validation")
print(f"three classes: discovery {np.bincount(y3[tr3]).tolist()}, validation {np.bincount(y3[te3]).tolist()} (healthy, TB, cancer)")

def run_mc(Xk, yk, tr, te, k, label, names):
    kp = mn.threshold_selection(Xk[tr], yk[tr], k, random_state=2024); m = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xk[tr][:, kp], yk[tr])
    p = m.predict(Xk[te][:, kp]); pr = m.predict_proba(Xk[te][:, kp])
    cvk = cross_val_score(Pipeline([("select", mn.GiniSelector(k, 2024)), ("forest", RandomForestClassifier(n_estimators=100, random_state=42))]), Xk[tr], yk[tr], cv=StratifiedKFold(10, shuffle=False))
    auc = roc_auc_score(yk[te], pr, multi_class="ovr", average="macro") if len(set(yk)) > 2 else roc_auc_score(yk[te], pr[:, 1]); counts = np.bincount(yk[te], minlength=max(yk) + 1)
    return dict(problem=label, k_selected=int(kp.sum()), n_train=len(tr), n_test=len(te), cv_accuracy=cvk.mean(), cv_sd=cvk.std(), test_accuracy=accuracy_score(yk[te], p), test_balanced_accuracy=balanced_accuracy_score(yk[te], p),
                test_macro_f1=f1_score(yk[te], p, average="macro"), test_auc_ovr=auc, test_majority_baseline=counts.max() / counts.sum(), confusion=str(confusion_matrix(yk[te], p, labels=sorted(set(yk))).tolist()), classes=";".join(names)), kp

mc = []
for k in (7, 50):
    r, kp = run_mc(X3, y3, tr3, te3, k, f"3 classes healthy/TB/LC, k={k}", NAMES); mc.append(r)
    if k == 7:
        r["selected_descriptors"] = ";".join(desc(int(i)) for i in np.flatnonzero(kp))
for a, b_ in ((1, 2), (0, 1), (0, 2)):
    m_ = np.isin(y3, [a, b_]); yb = (y3[m_] == b_).astype(int); coh = df3.cohort.to_numpy()[m_]
    r, _ = run_mc(X3[m_], yb, np.flatnonzero(coh == "discovery"), np.flatnonzero(coh == "validation"), 6, f"2 classes {NAMES[a]} vs {NAMES[b_]}, k=6 (7 bins)", [NAMES[a], NAMES[b_]]); mc.append(r)
rng3 = np.random.RandomState(2024); null3 = []
for _ in range(100):
    yp = y3.copy(); yp[tr3] = rng3.permutation(y3[tr3]); kp = mn.threshold_selection(X3[tr3], yp[tr3], 7, random_state=2024)
    null3.append(balanced_accuracy_score(y3[te3], RandomForestClassifier(n_estimators=100, random_state=42).fit(X3[tr3][:, kp], yp[tr3]).predict(X3[te3][:, kp])))
null3 = np.array(null3); mc[0].update(permutation_null_mean_balanced=null3.mean(), permutation_null_max_balanced=null3.max(), permutation_p=(1 + (null3 >= mc[0]["test_balanced_accuracy"]).sum()) / 101)
multiclass = pd.DataFrame(mc); display(multiclass[["problem", "k_selected", "cv_accuracy", "test_accuracy", "test_balanced_accuracy", "test_macro_f1", "test_auc_ovr", "test_majority_baseline", "confusion"]].round(3))
multiclass.to_csv(OUT / "lung_multiclass_summary.csv", index=False, float_format="%.6f")
m50, m7 = multiclass.iloc[1], multiclass.iloc[0]

three classes: discovery [37, 30, 55], validation [63, 63, 88] (healthy, TB, cancer)


,problem,k_selected,cv_accuracy,test_accuracy,test_balanced_accuracy,test_macro_f1,test_auc_ovr,test_majority_baseline,confusion
0,"3 classes healthy/TB/LC, k=7",8,0.876,0.682,0.640,0.558,0.836,0.411,"[[0, 1, 62], [0, 58, 5], [0, 0, 88]]"
1,"3 classes healthy/TB/LC, k=50",51,0.917,0.855,0.851,0.853,0.963,0.411,"[[42, 2, 19], [0, 63, 0], [10, 0, 78]]"
2,"2 classes TB vs cancer, k=6 (7 bins)",7,1.000,1.000,1.000,1.000,1.000,0.583,"[[63, 0], [0, 88]]"
3,"2 classes healthy vs TB, k=6 (7 bins)",7,0.986,0.944,0.944,0.944,0.997,0.500,"[[62, 1], [6, 57]]"
4,"2 classes healthy vs cancer, k=6 (7 bins)",7,0.889,0.755,0.727,0.732,0.887,0.583,"[[35, 28], [9, 79]]"


## 5. The bins of the first configuration

The regions of the first version of Table 3 came from the seven most important descriptors of a 1 000-tree
forest (seed 0) fitted on the discovery cohort: bin 227 (m/z [504,506), RT [2,4) min, both descriptors)
and bin 216 (m/z [482,484)). Three figures written to `results/lung/figures/` (`nb_bin227`, `nb_bin216`, `nb_bin131`; they are not part of the Supporting
Information) plot the normalized intensity against the stored m/z of the bin maximum in the validation
cohort for bins 227, 216 and 131 (the phenylalanylphenylalanine bin), and a fourth (`nb_phephe_boxplot_discovery`) shows the
intensity of the phenylalanylphenylalanine bin by class in the discovery cohort. We also rank the phenylalanylphenylalanine bin across
several forests and by its univariate AUC (Supporting Information, section 6.1).

In [9]:
imp1000 = RandomForestClassifier(n_estimators=1000, random_state=0, n_jobs=4).fit(X[train], y[train]).feature_importances_
top7 = np.argsort(imp1000)[::-1][:7]; print("seven most important descriptors (1 000 trees, seed 0):", [f"{desc(i)} rank {k + 1}" for k, i in enumerate(top7)])

def cluster_table(b, target, idx, eps=0.01):
    co = mn.bin_coordinates(X, b, GRID, components=("intensity", "mz")); present = co.intensity.to_numpy() > 0
    kp = mn.mass_cluster(co.mz.to_numpy(), target, present, eps=eps) & np.isin(np.arange(len(y)), idx)
    return dict(bin=b, bin_limits=mn.bin_label(b, GRID), target_mz=target, n_cluster=int(kp.sum()), tuberculosis=int((kp & (y == 0)).sum()), cancer=int((kp & (y == 1)).sum()),
                mean_mz=co.mz[kp].mean(), sd_ppm=co.mz[kp].std(ddof=0) / co.mz[kp].mean() * 1e6)
old_t3 = pd.DataFrame([dict(hypothesis="505.296 (26-hydroxycholesterol-3-sulfate [M+Na]+, withdrawn)", rank_I_1000t=mn.rank_of(imp1000, 227), rank_mz_1000t=mn.rank_of(imp1000, 227 + NB), **cluster_table(227, 505.2983, test)),
                       dict(hypothesis="505.335 (PPG-8 [M+Na]+)", rank_I_1000t=mn.rank_of(imp1000, 227), rank_mz_1000t=mn.rank_of(imp1000, 227 + NB), **cluster_table(227, 505.3349, test)),
                       dict(hypothesis="483.13 (silybin [M+H]+, withdrawn)", rank_I_1000t=mn.rank_of(imp1000, 216), rank_mz_1000t=mn.rank_of(imp1000, 216 + NB), **cluster_table(216, 483.1280, test))])
display(old_t3.round(4)); old_t3.to_csv(OUT / "first_configuration_bins.csv", index=False)
# nb_bin227, nb_bin216, nb_bin131 (validation cohort) and nb_phephe_boxplot_discovery (discovery cohort, phenylalanylphenylalanine bin): figures of the notebook, not part of the Supporting Information
def scatter(b, idx, name, xlim=None):
    co = mn.bin_coordinates(X, b, GRID, components=("intensity", "mz")); fig, ax = plt.subplots(figsize=(7, 5))
    for lab, col, nm in ((1, "tab:blue", "Lung Cancer"), (0, "tab:orange", "Pulmonary Tuberculosis")):
        s = idx[y[idx] == lab]; ax.scatter(co.mz[s], co.intensity[s], c=col, edgecolor="k", s=60, alpha=.8, label=nm)
    ax.set_xlabel("m/z"); ax.set_ylabel("Normalized intensity"); ax.grid(True, ls="--", alpha=.5); ax.legend(title="Category", loc="upper left"); ax.set_title(f"bin {b} ({mn.bin_label(b, GRID)}), validation cohort n = {len(idx)}", fontsize=9)
    if xlim: ax.set_xlim(*xlim)
    plt.tight_layout(); fig.savefig(FIG / f"{name}_bin{b}.pdf"); fig.savefig(FIG / f"{name}_bin{b}.png", dpi=150); plt.close(fig)
scatter(227, test, "nb"); scatter(216, test, "nb"); scatter(131, test, "nb")
co131 = mn.bin_coordinates(X, 131, GRID, components=("intensity", "mz")); same = np.abs(co131.mz - 313.15467) <= 313.15467 * 1e-5
fig, ax = plt.subplots(figsize=(5, 4)); data = [co131.intensity[disc & lc & same], co131.intensity[disc & ~lc & same]]
bp = ax.boxplot(data, widths=.5, patch_artist=True, showfliers=False); [p.set_facecolor(c) for p, c in zip(bp["boxes"], ("tab:blue", "tab:orange"))]
ax.set_xticks([1, 2]); ax.set_xticklabels([f"Lung Cancer (n={len(data[0])})", f"Pulmonary Tuberculosis (n={len(data[1])})"]); ax.set_ylabel("Normalized intensity"); ax.set_title("phenylalanylphenylalanine cluster (313.1542), discovery cohort", fontsize=9)
plt.tight_layout(); fig.savefig(FIG / "nb_phephe_boxplot_discovery.pdf"); fig.savefig(FIG / "nb_phephe_boxplot_discovery.png", dpi=150); plt.close(fig)
# phenylalanylphenylalanine intensity descriptor: univariate AUC with the first-configuration normalization, and its Gini rank across forests
auc_phe = {lab: roc_auc_score(y[idx], X[idx, 131]) for lab, idx in (("discovery", train), ("validation", test))}
print(f"phenylalanylphenylalanine bin (bin 131): AUC of the normalized intensity {auc_phe['discovery']:.3f} (discovery), {auc_phe['validation']:.3f} (validation)")
rank_rows = []
def phe_rank(Xs, ys, cols, trees, seed, label):
    f = RandomForestClassifier(n_estimators=trees, random_state=seed, n_jobs=1).fit(Xs[:, cols], ys); imp = f.feature_importances_; j = list(cols).index(131)
    rank_rows.append(dict(forest=label, trees=trees, seed=seed, importance=imp[j], rank=int(np.flatnonzero(np.argsort(imp)[::-1] == j)[0]) + 1, n_descriptors=len(cols), nonzero_descriptors=int((imp > 0).sum())))
allc = np.arange(2 * NB)
for s in range(3):
    phe_rank(X[train], y[train], allc, 1000, s, "discovery, all descriptors")
phe_rank(X[train], y[train], allc, 100, 2024, "discovery, selector forest of the classifier"); phe_rank(X[train], y[train], allc, 10000, 0, "discovery, all descriptors"); phe_rank(X, y, allc, 1000, 0, "all 236 samples")
auc_rows = []
for label, idx in (("discovery (n=85)", train), ("validation (n=151)", test), ("all (n=236)", np.arange(len(y)))):
    r_ = np.apply_along_axis(rankdata, 0, X[idx]); n1 = y[idx].sum(); n0 = len(idx) - n1; a_ = (r_[y[idx] == 1].sum(0) - n1 * (n1 + 1) / 2) / (n1 * n0); a2 = np.maximum(a_, 1 - a_)
    rk = np.empty(len(a2), int); rk[np.argsort(a2)[::-1]] = np.arange(1, len(a2) + 1)
    auc_rows.append(dict(cohort=label, phephe_auc=a_[131], rank_by_auc=int(rk[131]), descriptors_auc_above_0_83=int((a2 >= 0.83).sum()), descriptors_auc_1=int((a2 >= 0.999).sum())))
phe_auc_rank = pd.DataFrame(auc_rows); display(phe_auc_rank.round(3)); phe_auc_rank.to_csv(OUT / "lung_phephe_auc_rank.csv", index=False)

seven most important descriptors (1 000 trees, seed 0): ['mz[bin 818] rank 1', 'mz[bin 227] rank 2', 'I[bin 1380] rank 3', 'I[bin 1202] rank 4', 'mz[bin 216] rank 5', 'I[bin 227] rank 6', 'I[bin 1221] rank 7']


,hypothesis,rank_I_1000t,rank_mz_1000t,bin,bin_limits,target_mz,n_cluster,tuberculosis,cancer,mean_mz,sd_ppm
0,"505.296 (26-hydroxycholesterol-3-sulfate [M+Na]+, withdrawn)",6,2,227,"m/z [504,506) RT [2,4) min",505.2983,11,11,0,505.2900,30.0031
1,505.335 (PPG-8 [M+Na]+),6,2,227,"m/z [504,506) RT [2,4) min",505.3349,88,0,88,505.3348,2.4554
2,"483.13 (silybin [M+H]+, withdrawn)",226,5,216,"m/z [482,484) RT [2,4) min",483.1280,3,3,0,483.1335,3.8874


phenylalanylphenylalanine bin (bin 131): AUC of the normalized intensity 0.829 (discovery), 0.720 (validation)


,cohort,phephe_auc,rank_by_auc,descriptors_auc_above_0_83,descriptors_auc_1
0,discovery (n=85),0.829,594,590,30
1,validation (n=151),0.720,1019,512,43
2,all (n=236),0.648,1568,367,19


Downstream inspection of the ten most important bins of the selector forest (importance = maximum over
the intensity and m/z descriptors of the bin): occupancy, dispersion of the retained m/z (the RT block
is not part of this representation), dominant coordinate group (10 ppm), sub-clusters and flag.

In [10]:
imp_sel = RandomForestClassifier(n_estimators=100, random_state=2024).fit(X[train], y[train]).feature_importances_
per_bin = imp_sel.reshape(2, NB).max(axis=0); order = np.argsort(per_bin)[::-1][:10]
insp = pd.DataFrame([dict(**mn.inspect_bin(mn.bin_coordinates(X, int(b), GRID, components=("intensity", "mz")), y, int(b), GRID, ("TB", "cancer"), 10e-6, 30.0), importance=float(per_bin[b]), rank=k + 1) for k, b in enumerate(order)])
insp.to_csv(OUT / "downstream_inspection_lung.csv", index=False, float_format="%.6g")
display(insp[["rank", "bin", "bin_limits", "occupancy", "mz_sd_ppm", "support_D", "consistency_C", "n_mz_subclusters", "class_counts_present", "flag"]].round(3))
single = insp[insp.consistency_C >= 0.9]

,rank,bin,bin_limits,occupancy,mz_sd_ppm,support_D,consistency_C,n_mz_subclusters,class_counts_present,flag
0,1,874,"m/z [848,850) RT [4,6) min",1.0,356.294,0.288,0.288,5,93 TB/143 cancer,mixed column: sub-cluster before any chemical reading
1,2,818,"m/z [736,738) RT [4,6) min",1.0,481.302,0.508,0.508,6,93 TB/143 cancer,mixed column: sub-cluster before any chemical reading
2,3,545,"m/z [190,192) RT [4,6) min",1.0,830.354,0.318,0.318,4,93 TB/143 cancer,mixed column: sub-cluster before any chemical reading
3,4,1409,"m/z [968,970) RT [6,8) min",1.0,432.048,0.453,0.453,2,93 TB/143 cancer,mixed column: sub-cluster before any chemical reading
4,5,256,"m/z [562,564) RT [2,4) min",1.0,762.062,0.542,0.542,3,93 TB/143 cancer,mixed column: sub-cluster before any chemical reading
5,6,813,"m/z [726,728) RT [4,6) min",1.0,605.021,0.106,0.106,8,93 TB/143 cancer,mixed column: sub-cluster before any chemical reading
6,7,25,"m/z [100,102) RT [2,4) min",1.0,1.815,0.983,0.983,1,93 TB/143 cancer,single dominant ion (m/z only; RT not retained): proceed to XIC / ...
7,8,751,"m/z [602,604) RT [4,6) min",1.0,636.448,0.394,0.394,5,93 TB/143 cancer,mixed column: sub-cluster before any chemical reading
8,9,778,"m/z [656,658) RT [4,6) min",1.0,571.534,0.398,0.398,5,93 TB/143 cancer,mixed column: sub-cluster before any chemical reading
9,10,1378,"m/z [906,908) RT [6,8) min",1.0,423.065,0.360,0.360,10,93 TB/143 cancer,mixed column: sub-cluster before any chemical reading


## 6. Second configuration: stable bins and probabilistic quotient normalization

Two weaknesses of the first configuration show up in the stored coordinates. The per-sample maximum used
for the normalization is a single ion (lysoPC 16:0, m/z 496.34, in most sera), and an intensity descriptor
only compares the same ion across samples when that ion is the retained maximum of its bin in every sample. The second
configuration therefore (i) keeps the bins whose maximum lies within 10 ppm of the modal m/z of the
bin in at least 90 % of the 85 discovery samples (and is non-empty in at least 90 % of them), without
looking at the labels, and (ii) divides the bin maxima by their per-sample sum and normalizes them by the
probabilistic quotient method, with the discovery median spectrum as reference. Selector, seeds, number
of descriptors, classifier and split are those of the first configuration; the validation sera are only
scored.

In [11]:
PPM10, THR = 10e-6, 0.9

def stability(M, I, rows, thr=THR, ppm=PPM10):
    present = I[rows] > 0; mode = np.full(NB, np.nan); share = np.zeros(NB); occupancy = present.mean(0)
    for b in range(NB):
        ok = present[:, b]
        if not ok.any():
            continue
        md_ = float(np.median(M[rows, b][ok])); mode[b] = md_; share[b] = float(np.mean(np.abs(M[rows, b] - md_) <= md_ * ppm))
    return (occupancy >= thr) & (share >= thr), mode, share, occupancy

def tic_proxy(I):
    return I / I.sum(1, keepdims=True) * 1e3

def pqn(S, ref_rows, quotient_cols=None):
    cols_ = np.arange(S.shape[1]) if quotient_cols is None else np.asarray(quotient_cols)
    ref = np.median(S[ref_rows][:, cols_], axis=0); ok = ref > 0
    q = np.median(S[:, cols_][:, ok] / ref[ok], axis=1); return S / q[:, None], q

def normalize(how, ref_rows, quotient_cols=None):
    if how == "max":
        return I
    S = tic_proxy(I)
    if how == "summax":
        return S
    if how == "median":
        return I / np.array([np.median(r[r > 0]) for r in I])[:, None]
    if how == "pqn":
        return pqn(S, ref_rows, quotient_cols)[0]
    raise ValueError(how)

def representation(how="pqn", mask_rows=None, thr=THR, quotient="all"):
    rows = train if mask_rows is None else mask_rows
    mask, mode, share, occ = stability(M, I, rows, thr); cols_w = np.flatnonzero(mask)
    J = normalize(how, rows, cols_w if quotient == "stable" else None)
    cols_ = np.concatenate([cols_w, NB + cols_w])
    return np.concatenate([J, X[:, NB:]], axis=1)[:, cols_], cols_, dict(mask=mask, mode=mode, share=share, occupancy=occ, n_bins=int(mask.sum()))

def dname(c):
    return f"{'I' if c < NB else 'mz'}:{c % NB}"

def fit_and_score(Xk, yk, tr, te, seed_select=2024, seed_forest=42, k=6):
    kp = mn.threshold_selection(Xk[tr], yk[tr], k, random_state=seed_select); m = RandomForestClassifier(n_estimators=100, random_state=seed_forest).fit(Xk[tr][:, kp], yk[tr])
    p = m.predict(Xk[te][:, kp]); pr = m.predict_proba(Xk[te][:, kp])[:, 1]
    return dict(keep=np.flatnonzero(kp), pred=p, proba=pr, accuracy=accuracy_score(yk[te], p), balanced_accuracy=balanced_accuracy_score(yk[te], p), auc=roc_auc_score(yk[te], pr), model=m)

def article_pipeline(Xk, yk, tr, te, cols_=None, seed_select=2024, seed_forest=42, k=6, n_boot=1000):
    cvk = cross_val_score(Pipeline([("select", mn.GiniSelector(k, seed_select)), ("forest", RandomForestClassifier(n_estimators=100, random_state=seed_forest))]), Xk[tr], yk[tr], cv=10)
    r = fit_and_score(Xk, yk, tr, te, seed_select, seed_forest, k); boot = [accuracy_score(*resample(yk[te], r["pred"], random_state=i)) for i in range(n_boot)]
    yt = yk[te]; tp = int(((yt == 1) & (r["pred"] == 1)).sum()); tn = int(((yt == 0) & (r["pred"] == 0)).sum()); fp = int(((yt == 0) & (r["pred"] == 1)).sum()); fn = int(((yt == 1) & (r["pred"] == 0)).sum())
    sel = [int(cols_[j]) for j in r["keep"]] if cols_ is not None else [int(j) for j in r["keep"]]
    return dict(cv10_mean=float(cvk.mean()), cv10_sd=float(cvk.std()), cv10_folds=cvk, test_accuracy=float(r["accuracy"]), test_balanced=float(r["balanced_accuracy"]), test_auc=float(r["auc"]),
                bootstrap_mean=float(np.mean(boot)) if n_boot else np.nan, bootstrap_sd=float(np.std(boot)) if n_boot else np.nan, tp=tp, tn=tn, fp=fp, fn=fn, sens_lc=tp / (tp + fn), sens_tb=tn / (tn + fp),
                wilson_acc=mn.wilson(tp + tn, len(yt)), wilson_lc=mn.wilson(tp, tp + fn), wilson_tb=mn.wilson(tn, tn + fp), selected=sel, selected_names=[dname(c) for c in sel], pred=r["pred"], proba=r["proba"], keep=r["keep"])

def gini_ranking(Xk, yk, tr, trees=1000, seed=0):
    f = RandomForestClassifier(n_estimators=trees, random_state=seed, n_jobs=-1).fit(Xk[tr], yk[tr]); imp = f.feature_importances_
    rk = np.empty(len(imp), int); rk[np.argsort(imp)[::-1]] = np.arange(1, len(imp) + 1); return imp, rk

def univariate_auc(Xk, yk, rows):
    r_ = np.apply_along_axis(rankdata, 0, Xk[rows]); n1 = yk[rows].sum(); n0 = len(rows) - n1
    return (r_[yk[rows] == 1].sum(0) - n1 * (n1 + 1) / 2) / (n1 * n0)

def bin_limits(b):
    return 50 + 2 * (b % NMZ), 52 + 2 * (b % NMZ), 2 + 2 * (b // NMZ), 4 + 2 * (b // NMZ)

In [12]:
# the two defects, measured on the stored coordinates
argmax_w = I.argmax(1); top_ion = pd.Series(np.round([float(M[i, b]) for i, b in enumerate(argmax_w)], 2)).value_counts().head(5)
print("ion holding the per-sample maximum (m/z: number of sera):", top_ion.to_dict())
mask_d, mode_d, share_d, occ_d = stability(M, I, train)
share_v = np.array([np.mean(np.abs(M[test, b] - mode_d[b]) <= mode_d[b] * PPM10) if np.isfinite(mode_d[b]) else 0 for b in range(NB)])
n_stable = int(mask_d.sum()); n_stable_val = int(((share_v >= 0.9) & mask_d).sum())
print(f"bins with the same retained maximum in >= 90 % of the discovery samples: {n_stable} of {NB}; among them, same retained maximum in >= 90 % of the validation samples: {n_stable_val}")
NUM.update(max_ion_top=[(float(k), int(v)) for k, v in top_ion.items()], n_bins=NB, n_stable=n_stable, n_stable_val=n_stable_val)

ion holding the per-sample maximum (m/z: number of sera): {496.34: 220, 338.34: 11, 200.24: 2, 184.07: 2, 535.16: 1}
bins with the same retained maximum in >= 90 % of the discovery samples: 515 of 2375; among them, same retained maximum in >= 90 % of the validation samples: 271


In [13]:
Xk, cols, info = representation(); stable_w = np.flatnonzero(info["mask"]); cm = {int(c): i for i, c in enumerate(cols)}
config2 = article_pipeline(Xk, y, train, test, cols); sel_w = [c % NB for c in config2["selected"]]
print(f"second configuration: {info['n_bins']} stable bins ({len(cols)} descriptors); CV10 {config2['cv10_mean']:.3f} ± {config2['cv10_sd']:.3f} (folds {np.round(config2['cv10_folds'], 3)})")
print(f"validation: accuracy {config2['test_accuracy']:.4f} ({config2['tp'] + config2['tn']}/{len(test)}; Wilson 95 % {config2['wilson_acc'][0]:.3f}-{config2['wilson_acc'][1]:.3f}), balanced {config2['test_balanced']:.3f}, AUC {config2['test_auc']:.4f}, bootstrap {config2['bootstrap_mean']:.3f} ± {config2['bootstrap_sd']:.3f}")
print(f"LC {config2['tp']}/{config2['tp'] + config2['fn']} correct, TB {config2['tn']}/{config2['tn'] + config2['fp']} correct; selected descriptors {config2['selected_names']}; validation-day retained-maximum share {dict((dname(c), round(float(share_v[c % NB]), 2)) for c in config2['selected'])}")
comparison = pd.DataFrame([dict(configuration="source study (Chen et al. 2023)", cv10="--", accuracy=0.82, auc=0.89),
                           dict(configuration="first configuration: all bins, max normalization (as submitted)", cv10=f"{cv.mean():.2f} ± {cv.std():.2f}", accuracy=round(sc["test_accuracy"], 3), auc=round(sc["test_auc"], 3)),
                           dict(configuration=f"second configuration: {info['n_bins']} stable bins, PQN", cv10=f"{config2['cv10_mean']:.2f} ± {config2['cv10_sd']:.2f}", accuracy=round(config2["test_accuracy"], 3), auc=round(config2["test_auc"], 3))])
display(comparison); comparison.to_csv(OUT / "config2_comparison.csv", index=False)
NUM.update(config2_bins=info["n_bins"], config2_cv=config2["cv10_mean"], config2_cv_sd=config2["cv10_sd"], config2_acc=config2["test_accuracy"], config2_correct=config2["tp"] + config2["tn"], config2_balanced=config2["test_balanced"], config2_auc=config2["test_auc"],
           config2_boot=config2["bootstrap_mean"], config2_boot_sd=config2["bootstrap_sd"], config2_tp=config2["tp"], config2_fn=config2["fn"], config2_tn=config2["tn"], config2_fp=config2["fp"], config2_wilson=config2["wilson_acc"], config2_desc=config2["selected_names"])

second configuration: 515 stable bins (1030 descriptors); CV10 0.951 ± 0.082 (folds [1.    1.    0.889 1.    1.    0.75  0.875 1.    1.    1.   ])
validation: accuracy 0.9536 (144/151; Wilson 95 % 0.907-0.977), balanced 0.960, AUC 0.9996, bootstrap 0.954 ± 0.018
LC 81/88 correct, TB 63/63 correct; selected descriptors ['I:25', 'I:184', 'I:213', 'I:662', 'I:1735', 'I:2340', 'I:2341']; validation-day retained-maximum share {'I:25': 1.0, 'I:184': 0.48, 'I:213': 0.03, 'I:662': 0.99, 'I:1735': 0.86, 'I:2340': 0.92, 'I:2341': 0.93}


,configuration,cv10,accuracy,auc
0,source study (Chen et al. 2023),--,0.820,0.89
1,"first configuration: all bins, max normalization (as submitted)",1.00 ± 0.00,1.000,1.00
2,"second configuration: 515 stable bins, PQN",0.95 ± 0.08,0.954,1.00


Controls of the second configuration (same protocols as in section 4; the stability filter and the PQN
reference do not use the labels, so they stay fixed inside the permutation loop), learning curve, fifteen
seed pairs, and the sensitivity to the normalization, to the stability threshold and to the samples used
for the filter (Tables S15, S19 and S27; the seed pairs are in `results/lung/config2_seeds.csv`).

In [14]:
t0 = time.time(); rng2 = np.random.RandomState(2024); rows = []
for b in range(500):
    yp = y.copy(); yp[train] = rng2.permutation(y[train]); r = fit_and_score(Xk, yp, train, test)
    rows.append(dict(perm=b, validation_accuracy=accuracy_score(y[test], r["pred"]), validation_auc=roc_auc_score(y[test], r["proba"])))
perm2 = pd.DataFrame(rows); p2_acc = (1 + (perm2.validation_accuracy >= config2["test_accuracy"]).sum()) / 501; p2_auc = (1 + (perm2.validation_auc >= config2["test_auc"]).sum()) / 501
cv_null2 = np.array([cross_val_score(Pipeline([("select", mn.GiniSelector(6, 2024)), ("forest", RandomForestClassifier(n_estimators=100, random_state=42))]), Xk[train], rng2.permutation(y[train]), cv=10).mean() for _ in range(100)])
p2_cv = (1 + (cv_null2 >= config2["cv10_mean"]).sum()) / 101
psum = dict(n_perm=500, null_acc_mean=perm2.validation_accuracy.mean(), null_acc_sd=perm2.validation_accuracy.std(ddof=0), null_acc_max=perm2.validation_accuracy.max(), null_auc_mean=perm2.validation_auc.mean(), null_auc_max=perm2.validation_auc.max(),
            p_accuracy=p2_acc, p_auc=p2_auc, n_perm_cv=100, cv_null_mean=cv_null2.mean(), cv_null_max=cv_null2.max(), p_cv=p2_cv)
fo2 = RandomForestClassifier(n_estimators=100, random_state=42).fit(Mo[train], y[train])
ro2 = dict(validation_accuracy=accuracy_score(y[test], fo2.predict(Mo[test])), validation_auc=roc_auc_score(y[test], fo2.predict_proba(Mo[test])[:, 1]), cv_accuracy=cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), Mo[train], y[train], cv=10).mean(), majority_baseline=baseline)
rngr = np.random.RandomState(2024); rand2 = np.array([accuracy_score(y[test], RandomForestClassifier(n_estimators=100, random_state=42).fit(Xk[train][:, kp], y[train]).predict(Xk[test][:, kp])) for kp in (rngr.choice(Xk.shape[1], 7, replace=False) for _ in range(500))])
rows = []
for n in (10, 15, 20, 30, 40, 50, 60, 70, 85):
    for rr in range(20):
        sub = train if n >= len(train) else train[next(StratifiedShuffleSplit(1, train_size=n, random_state=1000 * n + rr).split(Xk[train], y[train]))[0]]
        res = fit_and_score(Xk, y, sub, test); rows.append(dict(n_train=len(sub), n_train_pos=int(y[sub].sum()), repeat=rr, accuracy=res["accuracy"], balanced_accuracy=res["balanced_accuracy"], auc=res["auc"]))
        if n >= len(train):
            break
lc2_rep = pd.DataFrame(rows)
lc2 = lc2_rep.groupby("n_train").agg(n_train_pos=("n_train_pos", "first"), repeats=("repeat", "count"), accuracy_mean=("accuracy", "mean"), accuracy_sd=("accuracy", "std"), accuracy_min=("accuracy", "min"), balanced_mean=("balanced_accuracy", "mean"), auc_mean=("auc", "mean"), auc_sd=("auc", "std")).reset_index()
print(f"{time.time() - t0:.0f} s. permutation null: accuracy {psum['null_acc_mean']:.3f} ± {psum['null_acc_sd']:.3f} (max {psum['null_acc_max']:.3f}), AUC {psum['null_auc_mean']:.3f} (max {psum['null_auc_max']:.3f}), p = {p2_acc:.4f} / {p2_auc:.4f}; CV null {psum['cv_null_mean']:.3f} (max {psum['cv_null_max']:.3f}) p = {p2_cv:.4f}")
print(f"run order: {ro2['validation_accuracy']:.3f} / AUC {ro2['validation_auc']:.3f}; seven random stable descriptors: {rand2.mean():.3f} ± {rand2.std():.3f} (max {rand2.max():.3f})"); display(lc2.round(3))
perm2.to_csv(OUT / "config2_permutation_null.csv", index=False); pd.DataFrame(dict(cv_null_accuracy=cv_null2)).to_csv(OUT / "config2_permutation_cv_null.csv", index=False)
pd.DataFrame(dict(random7_validation_accuracy=rand2)).to_csv(OUT / "config2_random_control.csv", index=False); lc2_rep.to_csv(OUT / "config2_learning_curve_repeats.csv", index=False); lc2.to_csv(OUT / "config2_learning_curve.csv", index=False)
pd.DataFrame([dict(**psum, **{f"run_order_{k}": v for k, v in ro2.items()}, random7_mean=rand2.mean(), random7_sd=rand2.std(), random7_max=rand2.max())]).to_csv(OUT / "config2_controls_summary.csv", index=False)
NUM.update(perm=psum, run_order=ro2, random7=dict(mean=float(rand2.mean()), sd=float(rand2.std()), max=float(rand2.max())),
           learning_curve={str(int(r_.n_train)): dict(acc=float(r_.accuracy_mean), sd=float(0 if np.isnan(r_.accuracy_sd) else r_.accuracy_sd), auc=float(r_.auc_mean)) for r_ in lc2.itertuples()})

824 s. permutation null: accuracy 0.507 ± 0.070 (max 0.682), AUC 0.503 (max 0.861), p = 0.0020 / 0.0020; CV null 0.555 (max 0.689) p = 0.0099
run order: 0.583 / AUC 0.561; seven random stable descriptors: 0.550 ± 0.081 (max 0.947)


,n_train,n_train_pos,repeats,accuracy_mean,accuracy_sd,accuracy_min,balanced_mean,auc_mean,auc_sd
0,10,6,20,0.654,0.097,0.417,0.663,0.799,0.092
1,15,10,20,0.654,0.105,0.483,0.645,0.819,0.132
2,20,13,20,0.719,0.144,0.477,0.693,0.899,0.106
3,30,19,20,0.803,0.151,0.477,0.802,0.967,0.036
4,40,26,20,0.863,0.120,0.570,0.850,0.968,0.050
5,50,32,20,0.891,0.118,0.616,0.882,0.984,0.023
6,60,39,20,0.909,0.110,0.609,0.901,0.977,0.041
7,70,45,20,0.945,0.057,0.755,0.944,0.995,0.011
8,85,55,1,0.954,NaN,0.954,0.960,1.000,NaN


In [15]:
rows = []
for ss in (2024, 0, 1, 2, 3):
    for sf in (42, 0, 1):
        r = article_pipeline(Xk, y, train, test, cols, seed_select=ss, seed_forest=sf, n_boot=0)
        rows.append(dict(seed_select=ss, seed_forest=sf, cv10=round(r["cv10_mean"], 3), validation_accuracy=round(r["test_accuracy"], 3), validation_auc=round(r["test_auc"], 3), selected=";".join(r["selected_names"])))
seeds = pd.DataFrame(rows); seeds.to_csv(OUT / "config2_seeds.csv", index=False); display(seeds)
rows = []
for rows_name, rows_idx in (("discovery", train), ("all 236", np.arange(len(y)))):
    for thr in (0.8, 0.9, 0.95):
        for how, quot in (("max", "all"), ("summax", "all"), ("pqn", "all"), ("pqn", "stable"), ("median", "all")):
            Xs, cs_, inf = representation(how, rows_idx, thr, quot); r = article_pipeline(Xs, y, train, test, cs_, n_boot=0); imp_s, rank_s = gini_ranking(Xs, y, train); cms = {int(c): i for i, c in enumerate(cs_)}
            rows.append(dict(filter_samples=rows_name, threshold=thr, normalization=how + (" (quotient on stable bins)" if quot == "stable" else ""), n_bins=inf["n_bins"], cv10=round(r["cv10_mean"], 3), cv10_sd=round(r["cv10_sd"], 3),
                             validation_accuracy=round(r["test_accuracy"], 3), balanced=round(r["test_balanced"], 3), validation_auc=round(r["test_auc"], 3), phephe_rank=int(rank_s[cms[131]]) if 131 in cms else None, selected=";".join(r["selected_names"])))
XI = Xk[:, :len(stable_w)]; rI = article_pipeline(XI, y, train, test, cols[:len(stable_w)], n_boot=0); impI, rankI = gini_ranking(XI, y, train)
rows.append(dict(filter_samples="discovery", threshold=0.9, normalization="pqn, intensity block only", n_bins=len(stable_w), cv10=round(rI["cv10_mean"], 3), cv10_sd=round(rI["cv10_sd"], 3), validation_accuracy=round(rI["test_accuracy"], 3),
                 balanced=round(rI["test_balanced"], 3), validation_auc=round(rI["test_auc"], 3), phephe_rank=int(rankI[cm[131]]), selected=";".join(rI["selected_names"])))
sens2 = pd.DataFrame(rows); sens2.to_csv(OUT / "config2_sensitivity.csv", index=False); display(sens2.drop(columns=["selected"]))
sd_ = sens2[(sens2.filter_samples == "discovery") & (sens2.normalization != "pqn, intensity block only")]
by_norm = {how: [float(v) for v in sd_[sd_.normalization == how].validation_accuracy] for how in ("pqn", "max", "summax")}
print(f"discovery filter, thresholds 0.8/0.9/0.95: PQN {by_norm['pqn']}, max {by_norm['max']}, TIC proxy {by_norm['summax']}; AUC range {sd_.validation_auc.min():.2f}-{sd_.validation_auc.max():.2f}")
NUM.update(seeds=dict(acc_min=float(seeds.validation_accuracy.min()), acc_max=float(seeds.validation_accuracy.max()), acc_median=float(seeds.validation_accuracy.median()), auc_min=float(seeds.validation_auc.min()), auc_max=float(seeds.validation_auc.max())),
           sens_ranges=dict(disc_acc_min=float(sd_.validation_accuracy.min()), disc_acc_max=float(sd_.validation_accuracy.max()), disc_auc_min=float(sd_.validation_auc.min()), disc_auc_max=float(sd_.validation_auc.max()), by_norm=by_norm,
                            intensity_only=dict(acc=rI["test_accuracy"], auc=rI["test_auc"], cv=rI["cv10_mean"], selected=rI["selected_names"], phe_rank=int(rankI[cm[131]]))))

,seed_select,seed_forest,cv10,validation_accuracy,validation_auc,selected
0,2024,42,0.951,0.954,1.0,I:25;I:184;I:213;I:662;I:1735;I:2340;I:2341
1,2024,0,0.951,0.974,1.0,I:25;I:184;I:213;I:662;I:1735;I:2340;I:2341
2,2024,1,0.975,0.987,1.0,I:25;I:184;I:213;I:662;I:1735;I:2340;I:2341
3,0,42,1.000,0.974,1.0,I:25;I:184;I:213;I:493;I:946;I:1704;I:2341
4,0,0,0.988,1.000,1.0,I:25;I:184;I:213;I:493;I:946;I:1704;I:2341
5,0,1,1.000,1.000,1.0,I:25;I:184;I:213;I:493;I:946;I:1704;I:2341
6,1,42,0.976,0.974,1.0,I:25;I:184;I:213;I:502;I:542;I:1704;I:2341
7,1,0,0.989,0.993,1.0,I:25;I:184;I:213;I:502;I:542;I:1704;I:2341
8,1,1,0.989,0.993,1.0,I:25;I:184;I:213;I:502;I:542;I:1704;I:2341
9,2,42,0.953,0.987,1.0,I:25;I:184;I:213;I:662;I:1704;I:1735;I:2341


,filter_samples,threshold,normalization,n_bins,cv10,cv10_sd,validation_accuracy,balanced,validation_auc,phephe_rank
0,discovery,0.80,max,721,0.988,0.037,0.901,0.903,0.980,46
1,discovery,0.80,summax,721,0.965,0.073,0.914,0.908,0.973,17
2,discovery,0.80,pqn,721,0.988,0.037,0.901,0.883,0.993,14
3,discovery,0.80,pqn (quotient on stable bins),721,0.953,0.076,0.934,0.941,0.989,17
4,discovery,0.80,median,721,0.976,0.047,0.868,0.850,0.971,39
5,discovery,0.90,max,515,0.929,0.058,0.623,0.548,0.929,18
6,discovery,0.90,summax,515,0.975,0.050,0.662,0.694,0.800,7
7,discovery,0.90,pqn,515,0.951,0.082,0.954,0.960,1.000,7
8,discovery,0.90,pqn (quotient on stable bins),515,0.975,0.050,0.623,0.676,0.963,8
9,discovery,0.90,median,515,0.940,0.060,0.616,0.540,0.679,18


discovery filter, thresholds 0.8/0.9/0.95: PQN [0.901, 0.954, 0.881], max [0.901, 0.623, 0.464], TIC proxy [0.914, 0.662, 0.47]; AUC range 0.68-1.00


Ranking of the stable bins (1 000-tree forest, seed 0, discovery cohort), the literature bins, and
the sub-models that show which bins carry the classification (Tables S16 and S18; Table 4 of the main
text is assembled in section 7, once the identities are known). Figure 3 shows the phenylalanylphenylalanine bin.

In [16]:
def load_maf():
    if not mn.raw_available("maf"):
        return None
    p = next((mn.RAW / "maf").glob("m_MTBLS6990_LC-MS_untargeted*maf.tsv")); maf = pd.read_csv(p, sep="\t", dtype=str)
    maf = maf[["metabolite_identification", "chemical_formula", "modifications", "mass_to_charge", "retention_time"]].copy()
    maf["mz"] = pd.to_numeric(maf.mass_to_charge, errors="coerce"); maf["rt"] = pd.to_numeric(maf.retention_time, errors="coerce"); return maf.dropna(subset=["mz"])

def maf_matches(maf, mz, rt, ppm=PPM10, drt=0.5):
    if maf is None:
        return ""
    hit = maf[np.abs(maf.mz - mz) <= mz * ppm]
    return "; ".join(f"{h.metabolite_identification} {h.modifications} (MAF RT {h.rt:.2f}{'' if abs(h.rt - rt) <= drt else ', RT differs'})" for h in hit.itertuples())

imp2, rank2 = gini_ranking(Xk, y, train); auc_d = univariate_auc(Xk, y, train); auc_v = univariate_auc(Xk, y, test); maf = load_maf(); rows = []
for j in np.argsort(imp2)[::-1][:40]:
    c = int(cols[j]); b = c % NB; mz_lo, mz_hi, rt_lo, rt_hi = bin_limits(b); ok = I[:, b] > 0; rt_med = float(np.median(R[ok, b]))
    rows.append(dict(rank=int(rank2[j]), descriptor=dname(c), selected=c in config2["selected"], bin=b, bin_limits=f"[{mz_lo},{mz_hi}) x [{rt_lo},{rt_hi})", mode_mz=round(float(info["mode"][b]), 4), share_disc=round(float(info["share"][b]), 3),
                     share_val=round(float(share_v[b]), 3), rt_median=round(rt_med, 2), importance=round(float(imp2[j]), 4), ratio_LC_TB_disc=round(float(np.median(Xk[disc & lc, j]) / max(np.median(Xk[disc & ~lc, j]), 1e-12)), 2),
                     ratio_LC_TB_val=round(float(np.median(Xk[~disc & lc, j]) / max(np.median(Xk[~disc & ~lc, j]), 1e-12)), 2), auc_disc=round(float(auc_d[j]), 3), auc_val=round(float(auc_v[j]), 3), maf=maf_matches(maf, float(info["mode"][b]), rt_med)))
T40 = pd.DataFrame(rows); T40.to_csv(OUT / "config2_ranking_top40.csv", index=False); display(T40.drop(columns=["descriptor", "importance"]))
LIT = {131: "phenylalanylphenylalanine", 35: "Phe immonium (in-source)", 691: "LysoPE 18:0", 662: "C18:2-carnitine", 650: "C16-carnitine", 663: "C18:1-carnitine", 1059: "anhydroretinol", 698: "LPC 16:0", 710: "LPC 18:2", 711: "LPC 18:1", 734: "LPC 22:6"}
litt = pd.DataFrame([dict(bin=b, compound=n, stable=b in cm, gini_rank=int(rank2[cm[b]]) if b in cm else None, auc_disc=round(float(auc_d[cm[b]]), 3) if b in cm else None, auc_val=round(float(auc_v[cm[b]]), 3) if b in cm else None, selected=b in sel_w) for b, n in LIT.items()])
display(litt); litt.to_csv(OUT / "config2_literature_ranks.csv", index=False)
lpc_fam = [698, 689, 502, 945, 946, 542, 493, 699, 579, 512]; lpc_ranks = sorted(int(T40.set_index("bin").loc[b, "rank"]) for b in lpc_fam if b in T40.bin.values)
NUM.update(rank_phephe=int(rank2[cm[131]]), n_stable_desc=len(cols), literature_ranks={n: int(rank2[cm[b]]) for b, n in LIT.items() if b in cm})

,rank,selected,bin,bin_limits,mode_mz,share_disc,share_val,rt_median,ratio_LC_TB_disc,ratio_LC_TB_val,auc_disc,auc_val,maf
0,1,True,2341,"[932,934) x [10,12)",932.1439,0.965,0.927,11.19,2.12,1.80,0.974,0.977,
1,2,True,25,"[100,102) x [2,4)",100.0761,1.000,1.000,2.05,0.36,0.30,0.034,0.000,
2,3,True,184,"[418,420) x [2,4)",419.2610,0.929,0.483,2.81,0.47,0.22,0.023,0.054,
3,4,True,213,"[476,478) x [2,4)",477.3029,0.953,0.033,3.98,0.55,1.14,0.036,0.661,
4,5,False,1704,"[608,610) x [8,10)",609.9349,0.918,0.881,8.18,1.80,2.31,0.946,0.982,
5,6,True,2340,"[930,932) x [10,12)",931.6420,0.941,0.921,11.19,1.52,1.46,0.881,0.876,
6,7,False,131,"[312,314) x [2,4)",313.1542,0.965,0.987,2.35,1.90,1.67,0.901,0.773,Phenylalanylphenylalanine [M+K] (MAF RT 2.32); Phenylalanylphenyla...
7,8,False,502,"[104,106) x [4,6)",104.1075,0.988,1.000,5.79,1.22,1.14,0.828,0.742,
8,9,False,689,"[478,480) x [4,6)",478.3295,0.953,0.775,5.79,1.27,1.13,0.838,0.698,
9,10,False,698,"[496,498) x [4,6)",496.3407,1.000,1.000,5.79,1.27,1.14,0.819,0.742,


,bin,compound,stable,gini_rank,auc_disc,auc_val,selected
0,131,phenylalanylphenylalanine,True,7,0.901,0.773,False
1,35,Phe immonium (in-source),True,19,0.844,0.780,False
2,691,LysoPE 18:0,True,38,0.824,0.760,False
3,662,C18:2-carnitine,True,12,0.848,0.813,True
4,650,C16-carnitine,True,30,0.838,0.811,False
5,663,C18:1-carnitine,True,34,0.771,0.800,False
6,1059,anhydroretinol,True,17,0.893,0.664,False
7,698,LPC 16:0,True,10,0.819,0.742,False
8,710,LPC 18:2,True,25,0.824,0.694,False
9,711,LPC 18:1,True,29,0.851,0.681,False


In [17]:
UNK = [25, 2341, 2340, 1704, 184, 213]; LITW = [131, 691, 662, 650, 663, 1059, 698, 710, 711, 734]
def sub(ws, label, select=False):
    ws = [b for b in ws if b in cm]; cc = np.array([cm[b] for b in ws] + ([cm[b] + len(stable_w) for b in ws] if select else [])); Xs = Xk[:, cc]
    kp = mn.threshold_selection(Xs[train], y[train], 6, random_state=2024) if select else np.ones(len(cc), bool); _, _, _, s_ = mn.holdout(Xs, y, train, test, kp, seed=42)
    return dict(model=label, n_bins=len(ws), validation_accuracy=round(s_["test_accuracy"], 3), validation_auc=round(s_["test_auc"], 3), descriptors=";".join(dname(int(cols[cc[j]])) for j in np.flatnonzero(kp)))
drv = pd.DataFrame([sub([25], "early-eluting ion m/z 100.076 alone"), sub([2341, 2340], "the two doubly charged ions m/z 931.64 and 932.14 alone"), sub(UNK, "the six unidentified top bins, no selection"),
                    sub(list(stable_w), "all stable bins, article selector (= second configuration)", select=True), sub([b for b in stable_w if b not in UNK], "stable bins without the six unidentified top bins, article selector", select=True),
                    sub(LITW, "the ten identified metabolite bins, no selection"), sub([131], "phenylalanylphenylalanine bin alone")])
display(drv); drv.to_csv(OUT / "config2_drivers.csv", index=False); NUM["drivers"] = drv.to_dict("records")

,model,n_bins,validation_accuracy,validation_auc,descriptors
0,early-eluting ion m/z 100.076 alone,1,0.695,0.915,I:25
1,the two doubly charged ions m/z 931.64 and 932.14 alone,2,0.947,0.976,I:2341;I:2340
2,"the six unidentified top bins, no selection",6,0.974,1.000,I:25;I:2341;I:2340;I:1704;I:184;I:213
3,"all stable bins, article selector (= second configuration)",515,0.954,1.000,I:25;I:184;I:213;I:662;I:1735;I:2340;I:2341
4,"stable bins without the six unidentified top bins, article selector",509,0.642,0.829,I:131;I:542;I:662;I:689;I:711;I:735;I:750
5,"the ten identified metabolite bins, no selection",10,0.735,0.860,I:131;I:691;I:662;I:650;I:663;I:1059;I:698;I:710;I:711;I:734
6,phenylalanylphenylalanine bin alone,1,0.642,0.609,I:131


In [18]:
# Figure 3: the phenylalanylphenylalanine bin of the second configuration (stored m/z of the maximum against the PQN-normalized intensity; class distributions per acquisition day)
MZ_THEO = 313.15467; j = cm[131]; x131, mz131 = Xk[:, j], M[:, 131]; same131 = np.abs(mz131 - MZ_THEO) <= MZ_THEO * PPM10; BLUE, ORANGE = "#1f77b4", "#ff7f0e"; rows = []
fig, axes = plt.subplots(2, 2, figsize=(11, 8.0), gridspec_kw=dict(width_ratios=[1.25, 1]))
for r_, (tag, name, m_, day) in enumerate((("A", "Discovery cohort", disc, "2020-10-08"), ("B", "Validation cohort", ~disc, "2020-11-25"))):
    a, b_ = x131[m_ & lc], x131[m_ & ~lc]; U, p = mannwhitneyu(a, b_, alternative="two-sided"); auc = roc_auc_score(y[m_], x131[m_]); n_same, n_other = int((m_ & same131).sum()), int((m_ & ~same131).sum())
    rows.append(dict(cohort=name, day=day, n=int(m_.sum()), n_LC=len(a), n_TB=len(b_), n_phephe_ion=n_same, n_other_ion=n_other, median_LC=np.median(a), median_TB=np.median(b_), median_ratio=np.median(a) / np.median(b_), U=U, p_two_sided=p, auc=auc,
                     other_classes=f"{int((m_ & ~same131 & ~lc).sum())} TB, {int((m_ & ~same131 & lc).sum())} LC"))
    ax = axes[r_, 0]; lo, hi = MZ_THEO * (1 - 25e-6), MZ_THEO * (1 + 25e-6)
    for cls_mask, col, lab in ((~lc, ORANGE, "tuberculosis"), (lc, BLUE, "lung cancer")):
        s = m_ & cls_mask & same131; ax.scatter(mz131[s], x131[s], s=36, color=col, edgecolor="k", linewidth=0.4, alpha=0.85, label=f"{lab} (n = {int((m_ & cls_mask).sum())})")
        o = m_ & cls_mask & ~same131
        if o.any():
            ax.scatter(np.full(o.sum(), lo + (hi - lo) * 0.012), x131[o], marker="<", s=60, color=col, edgecolor="k", linewidth=0.5, zorder=4)
    ax.axvline(MZ_THEO, color="0.45", lw=0.8, ls="--"); ax.set_xlim(lo, hi); ax.ticklabel_format(axis="x", useOffset=False, style="plain"); ax.grid(alpha=0.3, ls=":")
    ax.set_xlabel("stored m/z of the retained maximum"); ax.set_ylabel("PQN-normalized intensity"); ax.set_title(f"{tag}  {name} ({day}, n = {int(m_.sum())}); phenylalanylphenylalanine ion retained in {n_same}", fontsize=9.5, loc="left")
    ax.text(MZ_THEO, ax.get_ylim()[1], "  313.1547", fontsize=8, va="top", color="0.3")
    if n_other:
        ax.text(lo + (hi - lo) * 0.035, ax.get_ylim()[1] * 0.97, f"\u25c4 {n_other} sample{'s' if n_other > 1 else ''}: another ion retained\n   (m/z 312.22 or 313.02, off scale)", fontsize=7.5, va="top", color="0.25")
    ax.legend(fontsize=8, frameon=False, loc="upper right")
    ax = axes[r_, 1]; bp = ax.boxplot([a, b_], widths=0.55, patch_artist=True, showfliers=False, medianprops=dict(color="k"))
    for patch, col in zip(bp["boxes"], (BLUE, ORANGE)): patch.set_facecolor(col); patch.set_alpha(0.75)
    for k, d_ in enumerate((a, b_), 1): ax.scatter(np.random.RandomState(k).normal(k, 0.06, len(d_)), d_, s=7, color="k", alpha=0.35, zorder=3)
    top = max(np.percentile(a, 97), np.percentile(b_, 97)); h = top * 0.06; ax.plot([1, 1, 2, 2], [top + h, top + 2 * h, top + 2 * h, top + h], color="k", lw=1)
    ax.text(1.5, top + 2.3 * h, f"Mann\u2013Whitney U = {U:.0f}, p = {p:.1e} (two-sided)\nAUC = {auc:.2f}; ratio of medians = {np.median(a) / np.median(b_):.2f}", ha="center", va="bottom", fontsize=8.5)
    ax.set_ylim(top=top + 5.5 * h); ax.set_xticks([1, 2]); ax.set_xticklabels([f"lung cancer\n(n = {len(a)})", f"tuberculosis\n(n = {len(b_)})"]); ax.set_ylabel("PQN-normalized intensity"); ax.set_title("same bin, by class", fontsize=9.5, loc="left"); ax.grid(axis="y", alpha=0.3, ls=":")
plt.tight_layout(); fig.savefig(FIG / "fig3_phephe_bin.pdf"); fig.savefig(FIG / "fig3_phephe_bin.png", dpi=160); plt.close(fig)
phe_stats = pd.DataFrame(rows); display(phe_stats.round(4)); phe_stats.to_csv(OUT / "phephe_bin_stats.csv", index=False)

,cohort,day,n,n_LC,n_TB,n_phephe_ion,n_other_ion,median_LC,median_TB,median_ratio,U,p_two_sided,auc,other_classes
0,Discovery cohort,2020-10-08,85,55,30,82,3,1.3118,0.6912,1.8978,1486.0,0.0,0.9006,"3 TB, 0 LC"
1,Validation cohort,2020-11-25,151,88,63,149,2,4.3238,2.5960,1.6656,4287.0,0.0,0.7733,"1 TB, 1 LC"


In [19]:
# Figure S5 (PQN-normalized intensities by class and day) and two figures of the notebook that are not in the Supporting Information: nb_config2_ranking (ranking of the twenty most important stable bins) and nb_config2_controls (permutation null and learning curve)
top20 = T40.head(20); fig, ax = plt.subplots(figsize=(8.5, 6.2))
ax.barh(range(len(top20))[::-1], top20.importance, color=["#c0392b" if s else "#7f8c8d" for s in top20.selected]); ax.set_yticks(range(len(top20))[::-1])
ax.set_yticklabels([f"{r.rank}. m/z {r.mode_mz:.3f}  ({r.rt_median:.1f} min)" for r in top20.itertuples()], fontsize=8); ax.set_xlabel("Gini score (1 000 trees, discovery cohort)")
ax.set_title("Second lung configuration: twenty most important stable bins (red = selected descriptors)", fontsize=9); plt.tight_layout(); fig.savefig(FIG / "nb_config2_ranking.pdf"); fig.savefig(FIG / "nb_config2_ranking.png", dpi=160); plt.close(fig)
panels = [(b, f"m/z {info['mode'][b]:.3f}") for b in sel_w] + [(b, LIT[b]) for b in (131, 691, 650, 1059) if b not in sel_w]
fig, axes = plt.subplots(3, 4, figsize=(13, 8.5)); axes = axes.ravel()
for ax, (b, name) in zip(axes, panels):
    jj = cm[b]; data = [Xk[disc & lc, jj], Xk[disc & ~lc, jj], Xk[~disc & lc, jj], Xk[~disc & ~lc, jj]]; bp = ax.boxplot(data, widths=0.6, showfliers=False, patch_artist=True)
    for patch, col in zip(bp["boxes"], ["#1f77b4", "#ff7f0e", "#1f77b4", "#ff7f0e"]): patch.set_facecolor(col); patch.set_alpha(0.5)
    for k, d_ in enumerate(data, 1): ax.scatter(np.random.RandomState(k).normal(k, 0.07, len(d_)), d_, s=5, color="k", alpha=0.4)
    ax.set_xticks([1, 2, 3, 4]); ax.set_xticklabels(["LC\ndisc.", "TB\ndisc.", "LC\nval.", "TB\nval."], fontsize=7); ax.set_title(f"{name}\n{'selected' if b in sel_w else 'rank ' + str(int(rank2[jj]))}", fontsize=8); ax.tick_params(axis="y", labelsize=7)
for ax in axes[len(panels):]: ax.axis("off")
fig.suptitle("PQN-normalized retained maxima: blue = lung cancer, orange = tuberculosis; discovery (2020-10-08) and validation (2020-11-25) days", fontsize=9)
plt.tight_layout(); fig.savefig(FIG / "figS5_config2_boxplots.pdf"); fig.savefig(FIG / "figS5_config2_boxplots.png", dpi=160); plt.close(fig)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6)); a1.hist(perm2.validation_accuracy, bins=30, color="#7f8c8d"); a1.axvline(config2["test_accuracy"], color="#c0392b"); a1.set_xlabel("validation accuracy under permuted training labels"); a1.set_ylabel("permutations")
a1.set_title(f"whole-protocol permutation null (p = {p2_acc:.3f})", fontsize=9); a2.errorbar(lc2.n_train, lc2.accuracy_mean, yerr=lc2.accuracy_sd.fillna(0), marker="o", color="#2c3e50"); a2.set_xlabel("discovery samples used for selection and fitting"); a2.set_ylabel("validation accuracy"); a2.set_ylim(0.4, 1.02)
a2.set_title("learning curve (20 stratified subsamples per size)", fontsize=9); plt.tight_layout(); fig.savefig(FIG / "nb_config2_controls.pdf"); fig.savefig(FIG / "nb_config2_controls.png", dpi=160); plt.close(fig); print("Figures 3 and S5 written (with figS18 and figS19)")

Figures 3 and S5 written (with figS18 and figS19)


## 7. Raw data

### 7.1 The polypropylene-glycol series, the ions of the old Table 3 and the phenylalanylphenylalanine region (Tables S11–S13)

One pass per injection. The Waters MS$^E$ files hold two functions (1 = low-energy MS$^1$, 2 = high-energy
fragments). We look for the PPG oligomers n = 5–16 as [M+NH₄]⁺ and [M+Na]⁺ at their expected retention
times, for the two ions of the first version of Table 3 (505.296, 483.13), for the three non-PPG
descriptors of the first configuration, and for the phenylalanylphenylalanine ion (apex, [M+Na]⁺, isotopes, MS$^E$ immonium
and y₁ fragments, retained maximum of the window [312,314) × [2,4) min). The tolerance is ±15 ppm, this being a
Q-TOF at low intensity.

In [20]:
PPG_RT = {5: 2.77, 6: 2.97, 7: 3.19, 8: 3.46, 9: 3.78, 10: 4.14, 11: 4.55, 12: 4.95, 13: 5.38, 14: 5.80, 15: 6.22, 16: 6.64}
PHE = 313.15467; PPM_W = 15e-6; TARGETS = {}
for n_, rt_ in PPG_RT.items():
    TARGETS[f"ppg{n_}_NH4"] = (n_ * PPG_UNIT + PPG_END + NH4, rt_); TARGETS[f"ppg{n_}_Na"] = (n_ * PPG_UNIT + PPG_END + Na, rt_)
TARGETS.update({"ppg8_H": (8 * PPG_UNIT + PPG_END + H, 3.46), "tb_505.296": (505.2958, 3.43), "tb_483.13": (483.1286, 2.58), "d25_100.076": (100.0757, None), "d545_191.13": (191.128, None), "d813_726.5": (726.50, None)})

def injections():
    jobs = []
    for r in runs.itertuples():
        hits = [p for d_ in mn.RAW.glob("lung*") for p in d_.glob(f"{r.sample_name}.mzML")]
        if hits:
            jobs.append((str(hits[0]), "LC" if r.disease == "lung cancer" else "TB", r.cohort))
    return jobs

def process_lung(args):
    path, cls, cohort = args; ms1, ms2 = [], []; bpi = tic = 0.0; win = (0.0, np.nan, np.nan)
    for rt, mz, it in read_scans(path, 1.5, 7.5, 1, function=1):
        bpi = max(bpi, float(it.max()) if it.size else 0.0); tic += float(it.sum())
        row = dict(rt=rt, phe=peak(mz, it, PHE, PPM_W), phe_Na=peak(mz, it, PHE - H + Na, PPM_W), phe_M1=peak(mz, it, PHE + D13C, PPM_W))
        for k, (m, _) in TARGETS.items():
            row[k] = peak(mz, it, m, PPM_W)
        ms1.append(row)
        if 2.0 <= rt < 4.0:
            i, m = bin_maximum(mz, it, 312, 314)
            if i > win[0]:
                win = (i, m, rt)
    for rt, mz, it in read_scans(path, 1.5, 4.0, 2, function=2):
        ms2.append(dict(rt=rt, phe=peak(mz, it, PHE, PPM_W), imm=peak(mz, it, 120.08078, PPM_W), y1=peak(mz, it, 166.08626, PPM_W)))
    t, t2 = pd.DataFrame(ms1), pd.DataFrame(ms2)
    rec = dict(file=Path(path).stem, cls=cls, cohort=cohort, bpi_max=bpi, tic=tic, phe_win_I=win[0], phe_win_mz=round(win[1], 4) if win[1] == win[1] else np.nan, phe_win_rt=round(win[2], 2) if win[2] == win[2] else np.nan)
    for k, (m, rt0) in TARGETS.items():
        w = t if rt0 is None else t[(t.rt >= rt0 - 0.25) & (t.rt <= rt0 + 0.25)]
        if len(w) == 0 or w[k].max() <= 0:
            rec[f"{k}_I"] = 0.0; rec[f"{k}_rt"] = np.nan; continue
        a = w[k].idxmax(); rec[f"{k}_I"] = w[k].max(); rec[f"{k}_rt"] = round(t.rt[a], 2)
    rec["ppg_series_n"] = int(sum((rec[f"ppg{n_}_NH4_I"] > 300) and (rec[f"ppg{n_}_Na_I"] > 300) for n_ in range(6, 14)))
    w = t[(t.rt >= 3.0) & (t.rt <= 4.0)]; rec["ppg8_corr_NH4_Na"] = np.corrcoef(w.ppg8_NH4, w.ppg8_Na)[0, 1] if w.ppg8_NH4.std() > 0 and w.ppg8_Na.std() > 0 else np.nan
    w = t[(t.rt >= 2.0) & (t.rt < 4.0)]
    if w.phe.max() > 0:
        a = w.phe.idxmax(); apex_rt = t.rt[a]; p = t[(t.rt >= apex_rt - 0.08) & (t.rt <= apex_rt + 0.08)]; q = t2[(t2.rt >= apex_rt - 0.08) & (t2.rt <= apex_rt + 0.08)]
        rec.update(phe_apex_rt=round(apex_rt, 2), phe_apex_I=w.phe.max(), phe_scans=(w.phe > 0).sum(), phe_Na_rel=p.phe_Na.sum() / max(p.phe.sum(), 1), phe_Na_corr=np.corrcoef(w.phe, w.phe_Na)[0, 1] if w.phe_Na.std() > 0 else np.nan,
                   phe_M1_over_M=p.phe_M1.sum() / max(p.phe.sum(), 1), phe_mse_imm=q.imm.sum(), phe_mse_y1=q.y1.sum(), phe_mse_precursor=q.phe.sum())
    return rec

per_sample = OUT / "lung_raw_per_sample.csv"; d = mn.cached(per_sample)
if d is None and mn.raw_available("lung*"):
    jobs = injections(); t0 = time.time(); print(f"raw pass on {len(jobs)} injections ({sum(j[1] == 'LC' for j in jobs)} LC, {sum(j[1] == 'TB' for j in jobs)} TB)...", flush=True)
    d = pd.DataFrame(Parallel(n_jobs=6)(delayed(process_lung)(j) for j in jobs)); d.to_csv(per_sample, index=False); print(f"{time.time() - t0:.0f} s"); d = mn.cached(per_sample)
HAVE_LUNG_RAW = d is not None
print("per-sample raw table:", "available" if HAVE_LUNG_RAW else "absent (section 7.1 skipped)")

raw pass on 236 injections (143 LC, 93 TB)...


1067 s
per-sample raw table: available


In [21]:
if HAVE_LUNG_RAW:
    lc_, tb_ = d[d.cls == "LC"], d[d.cls == "TB"]; rows = []
    for n_ in PPG_RT:
        for ad in ("NH4", "Na"):
            k = f"ppg{n_}_{ad}_I"
            rows.append(dict(oligomer=n_, adduct=f"[M+{ad}]+", mz=TARGETS[f"ppg{n_}_{ad}"][0], expected_rt=PPG_RT[n_], LC_median=lc_[k].median(), LC_q25=lc_[k].quantile(.25), LC_q75=lc_[k].quantile(.75), LC_detected=(lc_[k] > 300).mean(),
                             TB_median=tb_[k].median(), TB_q25=tb_[k].quantile(.25), TB_q75=tb_[k].quantile(.75), TB_detected=(tb_[k] > 300).mean(), LC_over_TB=lc_[k].median() / max(tb_[k].median(), 1), p=mn.mwu(lc_[k], tb_[k])))
    ppg_series = pd.DataFrame(rows); ppg_series.to_csv(OUT / "lung_ppg_series.csv", index=False)
    def grp(label, g):
        return dict(group=label, n=len(g), complete_series=(g.ppg_series_n >= 6).sum(), series_n_median=g.ppg_series_n.median(), series_n_min=g.ppg_series_n.min(), ppg8_corr_NH4_Na=g.ppg8_corr_NH4_Na.median(), ppg_Na_sum_over_bpi=(sum(g[f"ppg{n_}_Na_I"] for n_ in range(6, 14)) / g.bpi_max).median())
    ppg_groups = pd.DataFrame([grp("LC", lc_), grp("TB", tb_)] + [grp(f"{g_} {coh}", dd[dd.cohort == coh]) for coh in ("discovery", "validation") for g_, dd in (("LC", lc_), ("TB", tb_))])
    ppg_groups.to_csv(OUT / "lung_ppg_groups.csv", index=False); display(ppg_groups.round(3))
    rows = []
    for k, lab in (("tb_505.296", "505.296 (26-hydroxycholesterol-3-sulfate [M+Na]+ hypothesis)"), ("tb_483.13", "483.13 (silybin [M+H]+ hypothesis)"), ("ppg8_Na", "505.335 (PPG-8 [M+Na]+)"), ("ppg8_H", "483.352 (PPG-8 [M+H]+)"),
                   ("d25_100.076", "100.076 (classifier descriptor, bin 25)"), ("d545_191.13", "191.13 (classifier descriptor, bin 545)"), ("d813_726.5", "726.5 (classifier descriptor, bin 813)")):
        rows.append(dict(ion=lab, LC_median=lc_[f"{k}_I"].median(), LC_q25=lc_[f"{k}_I"].quantile(.25), LC_q75=lc_[f"{k}_I"].quantile(.75), LC_detected=(lc_[f"{k}_I"] > 100).mean(), TB_median=tb_[f"{k}_I"].median(), TB_q25=tb_[f"{k}_I"].quantile(.25), TB_q75=tb_[f"{k}_I"].quantile(.75),
                         TB_detected=(tb_[f"{k}_I"] > 100).mean(), TB_over_LC=tb_[f"{k}_I"].median() / max(lc_[f"{k}_I"].median(), 1), p=mn.mwu(lc_[f"{k}_I"], tb_[f"{k}_I"]), rt_LC=lc_[f"{k}_rt"].median(), rt_TB=tb_[f"{k}_rt"].median()))
    t3_ions = pd.DataFrame(rows); t3_ions.to_csv(OUT / "lung_table3_ions.csv", index=False); display(t3_ions.round(3))
    phe_raw = pd.DataFrame([dict(quantity="313.155 detected in [2,4) min", LC=f"{lc_.phe_apex_I.notna().sum()}/{len(lc_)}", TB=f"{tb_.phe_apex_I.notna().sum()}/{len(tb_)}"),
                            dict(quantity="apex RT (median, sd)", LC=f"{lc_.phe_apex_rt.median():.2f} ({lc_.phe_apex_rt.std():.2f})", TB=f"{tb_.phe_apex_rt.median():.2f} ({tb_.phe_apex_rt.std():.2f})"),
                            dict(quantity="apex intensity, median [IQR]", LC=f"{lc_.phe_apex_I.median():.0f} [{lc_.phe_apex_I.quantile(.25):.0f}-{lc_.phe_apex_I.quantile(.75):.0f}]", TB=f"{tb_.phe_apex_I.median():.0f} [{tb_.phe_apex_I.quantile(.25):.0f}-{tb_.phe_apex_I.quantile(.75):.0f}]"),
                            dict(quantity="apex intensity, discovery / validation", LC=f"{lc_[lc_.cohort == 'discovery'].phe_apex_I.median():.0f} / {lc_[lc_.cohort == 'validation'].phe_apex_I.median():.0f}", TB=f"{tb_[tb_.cohort == 'discovery'].phe_apex_I.median():.0f} / {tb_[tb_.cohort == 'validation'].phe_apex_I.median():.0f}"),
                            dict(quantity="Mann-Whitney p (LC vs TB)", LC=f"{mn.mwu(lc_.phe_apex_I.fillna(0), tb_.phe_apex_I.fillna(0)):.1e}", TB=""),
                            dict(quantity="[M+Na]+ 335.137 relative to [M+H]+ (XIC correlation)", LC=f"{lc_.phe_Na_rel.median():.3f} ({lc_.phe_Na_corr.median():.2f})", TB=f"{tb_.phe_Na_rel.median():.3f} ({tb_.phe_Na_corr.median():.2f})"),
                            dict(quantity="M+1/M (expected 0.20)", LC=f"{lc_.phe_M1_over_M.median():.3f}", TB=f"{tb_.phe_M1_over_M.median():.3f}"),
                            dict(quantity="MSE at apex: immonium 120.081 / y1 166.086 / surviving precursor (summed counts)", LC=f"{lc_.phe_mse_imm.median():.0f} / {lc_.phe_mse_y1.median():.0f} / {lc_.phe_mse_precursor.median():.0f}", TB=f"{tb_.phe_mse_imm.median():.0f} / {tb_.phe_mse_y1.median():.0f} / {tb_.phe_mse_precursor.median():.0f}"),
                            dict(quantity="bin [312,314) x [2,4) retained maximum within 10 ppm of 313.1547", LC=f"{(np.abs(lc_.phe_win_mz - PHE) <= PHE * 1e-5).sum()}/{len(lc_)}", TB=f"{(np.abs(tb_.phe_win_mz - PHE) <= PHE * 1e-5).sum()}/{len(tb_)}")])
    phe_raw.to_csv(OUT / "lung_phephe.csv", index=False); display(phe_raw)
    g_ = ppg_groups.set_index("group")

,group,n,complete_series,series_n_median,series_n_min,ppg8_corr_NH4_Na,ppg_Na_sum_over_bpi
0,LC,143,143,8.0,7,0.903,0.051
1,TB,93,0,0.0,0,0.027,0.002
2,LC discovery,55,55,8.0,7,0.888,0.046
3,TB discovery,30,0,1.0,0,0.050,0.002
4,LC validation,88,88,8.0,8,0.914,0.055
5,TB validation,63,0,0.0,0,0.027,0.002


,ion,LC_median,LC_q25,LC_q75,LC_detected,TB_median,TB_q25,TB_q75,TB_detected,TB_over_LC,p,rt_LC,rt_TB
0,505.296 (26-hydroxycholesterol-3-sulfate [M+Na]+ hypothesis),551.000,360.209,1136.812,1.000,87.611,69.000,1651.214,0.409,0.159,0.000,3.26,3.41
1,483.13 (silybin [M+H]+ hypothesis),23.000,19.000,29.500,0.007,29.000,21.000,46.000,0.118,1.261,0.000,2.55,2.56
2,505.335 (PPG-8 [M+Na]+),5212.547,4298.016,8000.393,1.000,99.000,78.000,170.417,0.495,0.019,0.000,3.46,3.46
3,483.352 (PPG-8 [M+H]+),1782.440,1415.297,2337.155,1.000,51.401,42.000,71.000,0.086,0.029,0.000,3.46,3.45
4,"100.076 (classifier descriptor, bin 25)",37362.906,32202.266,43268.703,1.000,118458.125,92989.812,150329.250,1.000,3.170,0.000,2.05,2.05
5,"191.13 (classifier descriptor, bin 545)",62.000,43.500,151.372,0.392,62.000,50.000,399.999,0.312,1.000,0.028,5.43,5.97
6,"726.5 (classifier descriptor, bin 813)",97.015,85.000,114.000,0.455,225.000,194.000,256.729,0.989,2.319,0.000,6.35,6.27


,quantity,LC,TB
0,"313.155 detected in [2,4) min",143/143,93/93
1,"apex RT (median, sd)",2.34 (0.01),2.35 (0.01)
2,"apex intensity, median [IQR]",29346 [16441-39542],13987 [8372-25322]
3,"apex intensity, discovery / validation",16125 / 36239,9047 / 20541
4,Mann-Whitney p (LC vs TB),1.1e-10,
5,[M+Na]+ 335.137 relative to [M+H]+ (XIC correlation),0.012 (0.82),0.012 (0.73)
6,M+1/M (expected 0.20),0.184,0.182
7,MSE at apex: immonium 120.081 / y1 166.086 / surviving precursor (...,118318 / 1451 / 819,61242 / 743 / 415
8,"bin [312,314) x [2,4) retained maximum within 10 ppm of 313.1547",142/143,89/93


### 7.2 On the deposited vectors: PPG bins, masking experiment, phenylalanylphenylalanine ranks, source-study annotations (Table S14 and section 6.1 of the Supporting Information)

Bins whose lung-cancer retained maximum matches a PPG adduct (10 ppm); the article pipeline rerun with those
bins masked; the phenylalanylphenylalanine bin across forests and by univariate AUC; the MAF annotations of the source
study against the 1 000-tree Gini ranks; the forty most important descriptors; and the PPG retained maxima in the
QC and blank injections of the deposited table.

In [22]:
table_ppg = mn.ppg_adducts(3, 20); rows = []
for b in range(NB):
    mzs = M[lc, b]; r_ = pd.Series(np.round(mzs / 0.005) * 0.005).value_counts(); mode, share = float(r_.index[0]), r_.iloc[0] / len(mzs); hit = match_ppg(mode, 10e-6, table_ppg)
    if hit:
        rows.append(dict(bin=b, mz_lo=50 + 2 * (b % NMZ), rt_lo=2 + 2 * (b // NMZ), lc_retained_mz=mode, lc_share=share, lc_rt=np.median(R[lc, b]), ppg=hit, I_LC=np.median(I[lc, b]), I_TB=np.median(I[~lc, b]), tb_retained_mz=float(pd.Series(np.round(M[~lc, b] / 0.005) * 0.005).value_counts().index[0])))
ppg_w = pd.DataFrame(rows); ppg_w.to_csv(OUT / "lung_ppg_bins.csv", index=False)
strong = set(ppg_w[ppg_w.lc_share >= 0.5].bin); wide = set(ppg_w.bin) | {b + dd for b in ppg_w.bin for dd in (-1, 1)}
n_ppg_sel = sum((c % NB) in set(ppg_w.bin) for c in sel1)
print(f"{len(ppg_w)} bins with a PPG-matching lung-cancer retained maximum ({len(strong)} with share >= 0.5); {n_ppg_sel} of the 7 descriptors of the first configuration are PPG bins; stable set holds {len(set(stable_w) & strong)} strong / {len(set(stable_w) & set(ppg_w.bin))} any")
def run_mask(cols_, label):
    Xm = X[:, cols_]; cvm = cross_val_score(Pipeline([("select", mn.GiniSelector(6, 2024)), ("forest", RandomForestClassifier(n_estimators=100, random_state=42))]), Xm[train], y[train], cv=10)
    kp = mn.threshold_selection(Xm[train], y[train], 6, random_state=2024); _, _, _, s_ = mn.holdout(Xm, y, train, test, kp, seed=42)
    return dict(setting=label, n_bins_removed=NB - len(set(c % NB for c in cols_)), cv_accuracy=cvm.mean(), cv_sd=cvm.std(), test_accuracy=s_["test_accuracy"], test_auc=s_["test_auc"], descriptors=";".join(f"{['I', 'mz'][cols_[j] // NB]}:{cols_[j] % NB}" for j in np.flatnonzero(kp)))
masking = pd.DataFrame([run_mask(allc, "paper (all bins)"), run_mask(np.array([c for c in allc if c % NB not in strong]), "PPG-maximum bins removed (share >= 0.5)"), run_mask(np.array([c for c in allc if c % NB not in wide]), "PPG bins (any share) and m/z neighbours removed")])
masking.to_csv(OUT / "lung_ppg_masking.csv", index=False); display(masking.round(3))
phe_rank(X[train], y[train], np.array([c for c in allc if c % NB not in wide]), 1000, 0, "discovery, PPG bins and neighbours removed")
phe_ranks = pd.DataFrame(rank_rows); phe_ranks.to_csv(OUT / "lung_phephe_ranks.csv", index=False); display(phe_ranks)
NUM.update(stage1_ppg_desc=n_ppg_sel, ppg_strong_total=len(strong), ppg_strong_in_stable=len(set(stable_w) & strong), ppg_any_total=len(set(ppg_w.bin)), ppg_any_in_stable=len(set(stable_w) & set(ppg_w.bin)))

66 bins with a PPG-matching lung-cancer retained maximum (27 with share >= 0.5); 4 of the 7 descriptors of the first configuration are PPG bins; stable set holds 5 strong / 10 any


,setting,n_bins_removed,cv_accuracy,cv_sd,test_accuracy,test_auc,descriptors
0,paper (all bins),0,1.000,0.000,1.000,1.000,I:25;I:545;I:813;I:818;I:874;I:1409;mz:256
1,PPG-maximum bins removed (share >= 0.5),27,1.000,0.000,0.623,1.000,I:754;I:813;I:821;I:848;I:896;I:1221;I:1378;mz:1294
2,PPG bins (any share) and m/z neighbours removed,185,0.975,0.075,0.954,0.997,I:304;I:323;I:330;I:803;I:821;I:1209;mz:976


,forest,trees,seed,importance,rank,n_descriptors,nonzero_descriptors
0,"discovery, all descriptors",1000,0,0.000000,1778,4750,674
1,"discovery, all descriptors",1000,1,0.000230,288,4750,659
2,"discovery, all descriptors",1000,2,0.000000,1745,4750,601
3,"discovery, selector forest of the classifier",100,2024,0.000000,1565,4750,115
4,"discovery, all descriptors",10000,0,0.000000,2927,4750,2584
5,all 236 samples,1000,0,0.000051,493,4750,987
6,"discovery, PPG bins and neighbours removed",1000,0,0.000000,1754,4380,873


In [23]:
rank1000 = np.empty(X.shape[1], int); rank1000[np.argsort(imp1000)[::-1]] = np.arange(1, X.shape[1] + 1)
if maf is not None:
    maf2 = pd.read_csv(next((mn.RAW / "maf").glob("m_MTBLS6990_LC-MS_untargeted*maf.tsv")), sep="\t", dtype=str)[["chemical_formula", "metabolite_identification", "mass_to_charge", "modifications", "retention_time"]].copy()
    maf2["mz"] = pd.to_numeric(maf2.mass_to_charge, errors="coerce")
    def exact(row):
        try:
            m0 = mn.formula_mass(str(row.chemical_formula))
        except Exception:
            return row.mz
        add_ = {"[M+H]": H, "[M+Na]": Na, "[M+NH4]": NH4, "[M+K]": K}.get(str(row.modifications).strip())
        return m0 + add_ if add_ is not None and abs(m0 + add_ - row.mz) < 0.02 else row.mz
    maf2["mz_exact"] = maf2.apply(exact, axis=1); rows = []
    for r_ in maf2.itertuples():
        if not np.isfinite(r_.mz_exact):
            continue
        j = int((r_.mz_exact - 50) / 2)
        for rt_idx in range(5):
            b = rt_idx * NMZ + j; same_ = np.abs(M[:, b] - r_.mz_exact) <= r_.mz_exact * 10e-6
            if same_.mean() < 0.3:
                continue
            rows.append(dict(metabolite=r_.metabolite_identification, formula=r_.chemical_formula, adduct=r_.modifications, mz_exact=r_.mz_exact, maf_rt=r_.retention_time, bin=b, rt_interval=f"[{2 + 2 * rt_idx},{4 + 2 * rt_idx})", retained_share=same_.mean(), retained_rt=np.median(R[same_, b]),
                             gini1000_I=int(rank1000[b]), gini1000_mz=int(rank1000[NB + b]), I_LC_disc=np.median(I[lc & disc, b]), I_TB_disc=np.median(I[~lc & disc, b]), I_LC_val=np.median(I[lc & ~disc, b]), I_TB_val=np.median(I[~lc & ~disc, b]), ppg=match_ppg(r_.mz_exact, 10e-6, table_ppg)))
    maf_ranks = pd.DataFrame(rows).sort_values("gini1000_I"); maf_ranks.to_csv(OUT / "lung_maf_ranks.csv", index=False); display(maf_ranks.head(12))
    rows = []
    for pos, idx in enumerate(np.argsort(imp1000)[::-1][:40], 1):
        blk, b = divmod(int(idx), NB); wl = float(pd.Series(M[lc, b].round(3)).value_counts().index[0]); wt = float(pd.Series(M[~lc, b].round(3)).value_counts().index[0]); hit = maf2[np.abs(maf2.mz_exact - wl) <= wl * 10e-6]
        rows.append(dict(position=pos, block=["I", "mz"][blk], bin=b, mz_bin=f"[{50 + 2 * (b % NMZ)},{52 + 2 * (b % NMZ)})", rt_interval=f"[{2 + 2 * (b // NMZ)},{4 + 2 * (b // NMZ)})", retained_LC=wl, retained_TB=wt, retained_rt=np.median(R[:, b]), I_LC=np.median(I[lc, b]), I_TB=np.median(I[~lc, b]), ppg=match_ppg(wl, 10e-6, table_ppg), maf=";".join(hit.metabolite_identification.astype(str))))
    pd.DataFrame(rows).to_csv(OUT / "lung_top40_gini.csv", index=False)
else:
    print("MAF file not available: the annotation ranks (lung_maf_ranks.csv) and the top-40 list are skipped")
full = mn.lung_table(); Vf = np.stack(full.output_vector.to_list()); Mf = 50 + 950 * Vf[:, NB:2 * NB]; full["group"] = full.disease.fillna(full.sample_type)
rows = [dict(bin=b, ppg_ion=match_ppg(mz_, 10e-6, table_ppg), group=g_, n=len(gg), retained_is_ppg=int((np.abs(Mf[gg.index, b] - mz_) <= mz_ * 15e-6).sum())) for b, mz_ in ((227, 8 * PPG_UNIT + PPG_END + Na), (256, 9 * PPG_UNIT + PPG_END + Na), (216, 8 * PPG_UNIT + PPG_END + H)) for g_, gg in full.groupby("group")]
qc_blank = pd.DataFrame(rows); qc_blank.to_csv(OUT / "lung_ppg_qc_blank.csv", index=False); display(qc_blank)

,metabolite,formula,adduct,mz_exact,maf_rt,bin,rt_interval,retained_share,retained_rt,gini1000_I,gini1000_mz,I_LC_disc,I_TB_disc,I_LC_val,I_TB_val,ppg
15,"5,6-Dihydroxyprostaglandin F1a",C20H36O7,[M+H],389.253380,2.9643,169,"[2,4)",0.555085,2.973100,45,150,0.010031,0.001153,0.011517,0.001076,PPG6 [M+Na]+
54,LysoPC(20:4/0:0),C28H50NO7P,[M+H],544.339766,11.2756,722,"[4,6)",1.000000,5.264000,373,4732,0.119262,0.109015,0.113598,0.105900,
44,Tryptophol,C10H11NO,[M+H],184.073400,6.7907,542,"[4,6)",1.000000,5.793983,467,4604,0.314015,0.315916,0.339073,0.340580,
72,LysoPC(20:3/0:0),C28H52NO7P,[M+H],546.355416,6.8082,1198,"[6,8)",0.995763,7.070967,499,2905,0.015750,0.016801,0.017629,0.019252,
3,LysoPC(20:3/0:0),C28H52NO7P,[M+H],546.355416,6.7896,1198,"[6,8)",0.995763,7.070967,499,2905,0.015750,0.016801,0.017629,0.019252,
47,Tryptophol,C10H11NO,[M+H],184.073400,6.7907,1967,"[10,12)",1.000000,11.285042,562,3569,0.432807,0.556594,0.526238,0.623968,
22,LysoPC(16:0/0:0),C24H50NO6P,[M+Na],502.326795,5.303,1651,"[8,10)",0.656780,8.967183,760,2731,0.000501,0.000663,0.000900,0.001131,
31,Alanylleucine,C9H18N2O3,[M+K],241.094900,9.1688,1520,"[8,10)",0.300847,9.180616,795,2632,0.002985,0.004509,0.001471,0.001413,
16,"5,6-Dihydroxyprostaglandin F1a",C20H36O7,[M+H],389.253380,2.9643,1594,"[8,10)",0.550847,9.925950,825,2567,0.000608,0.000672,0.000914,0.000983,PPG6 [M+Na]+
46,Tryptophol,C10H11NO,[M+H],184.073400,6.7907,1492,"[8,10)",1.000000,9.315342,854,2584,0.242650,0.337288,0.269500,0.332395,


,bin,ppg_ion,group,n,retained_is_ppg
0,227,PPG8 [M+Na]+,Experimental blank,2,0
1,227,PPG8 [M+Na]+,Healthy Control,100,100
2,227,PPG8 [M+Na]+,Pulmonary Tuberculosis,93,0
3,227,PPG8 [M+Na]+,lung cancer,143,143
4,227,PPG8 [M+Na]+,pooled quality control sample,51,51
5,256,PPG9 [M+Na]+,Experimental blank,2,0
6,256,PPG9 [M+Na]+,Healthy Control,100,100
7,256,PPG9 [M+Na]+,Pulmonary Tuberculosis,93,0
8,256,PPG9 [M+Na]+,lung cancer,143,143
9,256,PPG9 [M+Na]+,pooled quality control sample,51,51


### 7.3 Identity of the ions retained by the stable bins (all 236 injections; Tables S16 and S17, Table 4)

For the forty most important stable bins and the PC 34:1 bin, in every injection: apex of the modal
ion inside the bin (low-energy function, ±15 ppm), isotope ratios at the apex (M+1, M+2 and the +0.502
spacing of doubly charged ions), relative intensity and scan-wise correlation of the generic and
hypothesis-specific companions, fragments of the MS$^E$ scan nearest to the apex and, because the
deposited vectors were binned from all Waters functions, the intensity of both functions at the
retention time of the stored bin maximum.

In [24]:
PPM15 = 15e-6; NA_, K_, NH4_, H_ = 22.989218, 38.963158, 18.033823, 1.007276
LPC16 = {"LPC 16:0 [M+H]+": 496.3398, "LPC 16:0 -H2O": 478.3292, "phosphocholine 184.073": 184.0733, "choline 104.107": 104.1070}
HYP = {
    2341: dict(name="unidentified doubly charged ion 932.144 (0.502 above 931.642)", companions={"931.642 (monoisotopic ?)": 931.6420}, frags={}),
    2340: dict(name="unidentified doubly charged ion 931.642", companions={"932.144 (+0.502)": 932.1439}, frags={}),
    25: dict(name="early-eluting ion 100.076 (C5H10NO+; not a valine/betaine fragment)", companions={"valine/betaine [M+H]+ 118.086": 118.0863, "choline 104.107": 104.1070}, frags={"58.065": 58.0651, "72.081": 72.0808, "69.034": 69.0335}),
    184: dict(name="unidentified ion 419.261 (M+1/M 0.09)", companions={}, frags={}),
    213: dict(name="unidentified ion 477.303 (M+1/M 0.10)", companions={}, frags={}),
    1704: dict(name="unidentified doubly charged ion 609.935", companions={}, frags={}),
    131: dict(name="phenylalanylphenylalanine [M+H]+", companions={"Phe immonium 120.081": 120.0808, "y1 166.086": 166.0863, "[M+Na]+ 335.137": 335.1366}, frags={"120.081": 120.0808, "166.086": 166.0863, "103.054": 103.0542}),
    35: dict(name="phenylalanine immonium 120.081 (in-source fragment of Phe)", companions={"phenylalanylphenylalanine 313.155": 313.1547, "Phe [M+H]+ 166.086": 166.0863}, frags={"103.054": 103.0542, "91.054": 91.0542}),
    502: dict(name="choline fragment 104.107 (LPC 16:0 in-source)", companions=LPC16, frags={}),
    689: dict(name="LPC 16:0 [M+H-H2O]+", companions=LPC16, frags={"184.073": 184.0733, "104.107": 104.1070}),
    698: dict(name="LPC 16:0 [M+H]+", companions={"LPC 16:0 -H2O": 478.3292, "phosphocholine 184.073": 184.0733, "choline 104.107": 104.1070, "[M+Na]+ 518.322": 518.3217}, frags={"184.073": 184.0733, "104.107": 104.1070, "86.096": 86.0964}),
    945: dict(name="LPC 16:0 [2M+H]+", companions=LPC16, frags={"496.340": 496.3398, "184.073": 184.0733}),
    946: dict(name="LPC 16:0 [2M+H]+ M+1", companions={"991.673 (monoisotopic)": 991.6723, **LPC16}, frags={}),
    542: dict(name="phosphocholine 184.073 (LPC 16:0 in-source)", companions=LPC16, frags={}),
    493: dict(name="86.097 (choline -H2O, LPC 16:0 in-source)", companions=LPC16, frags={}),
    699: dict(name="LPC 16:0 M+2 isotopologue", companions=LPC16, frags={}),
    579: dict(name="glycerophosphocholine 258.110 (LPC in-source)", companions=LPC16, frags={"184.073": 184.0733, "104.107": 104.1070}),
    512: dict(name="125.000 (cyclic ethylene phosphate, PC/LPC in-source)", companions=LPC16, frags={}),
    749: dict(name="PPG decamer [M+H]+ 599.440 (co-eluting [M+NH4]+/[M+Na]+)", companions=LPC16, frags={}),
    1701: dict(name="unidentified ion 603.222 (M+2/M 0.12)", companions={}, frags={}),
    662: dict(name="linoleoylcarnitine C18:2 [M+H]+", companions={"[M+Na]+ 446.324": 446.3241, "[M+H-59]+ 365.269": 365.2686, "85.028 in-source": 85.0284}, frags={"85.028": 85.0284, "60.081": 60.0808, "365.269": 365.2686, "linoleoyl 263.237": 263.2369}),
    650: dict(name="palmitoylcarnitine C16 [M+H]+", companions={"[M+Na]+ 422.324": 422.3241, "[M+H-59]+ 341.269": 341.2686, "85.028 in-source": 85.0284}, frags={"85.028": 85.0284, "60.081": 60.0808, "341.269": 341.2686, "palmitoyl 239.237": 239.2369}),
    663: dict(name="oleoylcarnitine C18:1 [M+H]+", companions={"[M+Na]+ 448.340": 448.3397, "[M+H-59]+ 367.284": 367.2843, "85.028 in-source": 85.0284}, frags={"85.028": 85.0284, "60.081": 60.0808, "367.284": 367.2843, "oleoyl 265.253": 265.2526}),
    492: dict(name="85.028 (acylcarnitine in-source fragment)", companions={"C16-carnitine 400.342": 400.3421, "C18:1-carnitine 426.358": 426.3578, "C18:2-carnitine 424.342": 424.3421}, frags={}),
    1735: dict(name="unidentified doubly charged ion 670.458", companions={}, frags={}),
    1059: dict(name="retinol [M+H-H2O]+ / anhydroretinol [M+H]+ (C20H29+)", companions={"retinol [M+H]+ 287.237": 287.2369, "retinol [M+Na]+ 309.219": 309.2189}, frags={"213.164": 213.1638, "157.101": 157.1012, "119.086": 119.0855, "95.086": 95.0855}),
    1937: dict(name="unidentified ion 124.087", companions={}, frags={}),
    165: dict(name="unidentified ion 381.130", companions={}, frags={}),
    1767: dict(name="unidentified doubly charged ion 735.995", companions={}, frags={}),
    710: dict(name="LPC 18:2 [M+H]+", companions={"[M+Na]+ 542.322": 542.3217, "-H2O 502.329": 502.3292, "phosphocholine 184.073": 184.0733}, frags={"184.073": 184.0733, "104.107": 104.1070}),
    721: dict(name="LPC 18:2 [M+Na]+", companions={"LPC 18:2 [M+H]+ 520.340": 520.3398, "phosphocholine 184.073": 184.0733}, frags={"184.073": 184.0733}),
    711: dict(name="LPC 18:1 [M+H]+", companions={"[M+Na]+ 544.337": 544.3374, "-H2O 504.345": 504.3449, "phosphocholine 184.073": 184.0733}, frags={"184.073": 184.0733, "104.107": 104.1070}),
    734: dict(name="LPC 22:6 [M+H]+", companions={"[M+Na]+ 590.322": 590.3217, "-H2O 550.329": 550.3292, "phosphocholine 184.073": 184.0733}, frags={"184.073": 184.0733, "104.107": 104.1070}),
    691: dict(name="LysoPE 18:0 [M+H]+", companions={"[M+Na]+ 504.306": 504.3061, "[M+H-141]+ 341.305": 341.3050}, frags={"341.305": 341.3050, "323.294": 323.2945}),
    1792: dict(name="PC 36:3 [M+H]+ (mass, 3 ppm)", companions={"[M+Na]+ 806.567": 806.5670, "phosphocholine 184.073": 184.0733}, frags={"184.073": 184.0733}),
    1466: dict(name="unidentified ion 133.086", companions={}, frags={}),
    1746: dict(name="unidentified doubly charged ion 692.471", companions={}, frags={}),
    1975: dict(name="unidentified ion 200.238", companions={}, frags={}),
    231: dict(name="unidentified ion 512.334", companions={}, frags={}),
    523: dict(name="unidentified ion 146.982", companions={}, frags={}),
    1305: dict(name="m/z 760.581 (PC 34:1 [M+H]+ within 6 ppm; not assigned)", companions={"[M+Na]+ 782.567": 782.5670, "phosphocholine 184.073": 184.0733}, frags={"184.073": 184.0733}),
}

def targets():
    tg = [dict(bin=int(r_.bin), rank=int(r_.rank), mz=float(r_.mode_mz), rt=float(r_.rt_median)) for r_ in T40.itertuples()]
    ok = I[:, 1305] > 0; tg.append(dict(bin=1305, rank=385, mz=float(np.median(M[ok, 1305])), rt=float(np.median(R[ok, 1305]))))
    for t_ in tg:
        b = t_["bin"]; t_["rt_dep_disc"] = float(np.median(R[disc, b])); t_["rt_dep_val"] = float(np.median(R[~disc, b]))
        mz_lo, mz_hi, rt_lo, rt_hi = bin_limits(b); hyp = HYP.get(b, dict(name=f"unidentified ion {t_['mz']:.3f}", companions={}, frags={}))
        t_.update(rt_lo=rt_lo, rt_hi=rt_hi, name=hyp["name"], companions=hyp["companions"], frags=hyp["frags"])
    return tg

def xic(scans, target, ppm=PPM15):
    out = np.zeros(len(scans)); lo, hi = target * (1 - ppm), target * (1 + ppm)
    for i, (_, mz, it) in enumerate(scans):
        a, b = np.searchsorted(mz, lo), np.searchsorted(mz, hi)
        if b > a:
            out[i] = it[a:b].max()
    return out

def peak_in(mz, it, target, ppm=PPM15):
    a, b = np.searchsorted(mz, target * (1 - ppm)), np.searchsorted(mz, target * (1 + ppm)); return float(it[a:b].max()) if b > a else 0.0

def process_identity(args):
    path, cls, cohort, tg = args
    f1 = [(rt, mz, it) for rt, mz, it in read_scans(path, 1.9, 12.1, 1, function=1)]; f2 = [(rt, mz, it) for rt, mz, it in read_scans(path, 1.9, 12.1, 2, function=2)]
    if f1 and np.any(np.diff(f1[0][1]) < 0):
        f1 = [(rt, mz[o], it[o]) for rt, mz, it in f1 for o in [np.argsort(mz)]]; f2 = [(rt, mz[o], it[o]) for rt, mz, it in f2 for o in [np.argsort(mz)]]
    rts1 = np.array([s[0] for s in f1]); rts2 = np.array([s[0] for s in f2]) if f2 else np.array([]); out = []
    for t_ in tg:
        sel = np.flatnonzero((rts1 >= t_["rt_lo"]) & (rts1 < t_["rt_hi"]))
        if not len(sel):
            continue
        scans = [f1[i] for i in sel]; x = xic(scans, t_["mz"]); j = int(np.argmax(x)); apex_rt, mz_a, it_a = scans[j]
        row = dict(sample=Path(path).stem, cls=cls, cohort=cohort, bin=t_["bin"], target=t_["name"], mz=t_["mz"], apex_rt=apex_rt, I_apex=x[j], detected=x[j] > 0, I_sum_interval=float(x.sum()))
        rt_dep = t_["rt_dep_disc"] if cohort == "discovery" else t_["rt_dep_val"]; row["rt_dep"] = rt_dep; row["I_f1_at_dep"] = float(x[int(np.argmin(np.abs(np.array([s[0] for s in scans]) - rt_dep)))])
        if len(rts2):
            sel2 = np.flatnonzero((rts2 >= t_["rt_lo"]) & (rts2 < t_["rt_hi"]))
            if len(sel2):
                scans2 = [f2[i] for i in sel2]; x2 = xic(scans2, t_["mz"]); j2 = int(np.argmax(x2)); rts2w = np.array([s[0] for s in scans2])
                row["apex_rt_f2"] = scans2[j2][0]; row["I_apex_f2"] = float(x2[j2]); kd = int(np.argmin(np.abs(rts2w - rt_dep))); row["I_f2_at_dep"] = float(x2[kd])
                _, mzd, itd = scans2[kd]; kp_ = mzd < t_["mz"] + 1.5; fr = sorted(zip(np.round(mzd[kp_], 3), itd[kp_]), key=lambda kv: -kv[1])[:8]; row["mse_top_at_dep"] = ";".join(f"{a}:{b:.0f}" for a, b in fr)
        if x[j] > 0:
            m = t_["mz"]; row["M1_over_M"] = peak_in(mz_a, it_a, m + 1.00336) / x[j]; row["M2_over_M"] = peak_in(mz_a, it_a, m + 2.00671) / x[j]; row["Mhalf_over_M"] = peak_in(mz_a, it_a, m + 0.50168) / x[j]
            for lab, dm in (("Na", NA_ - H_), ("K", K_ - H_), ("NH4", NH4_ - H_), ("H2O", -18.010565)):
                row[f"rel_{lab}"] = peak_in(mz_a, it_a, m + dm) / x[j]
            for lab, cmass in t_["companions"].items():
                row[f"rel_{lab}"] = peak_in(mz_a, it_a, cmass) / x[j]; xc = xic(scans, cmass); row[f"corr_{lab}"] = float(np.corrcoef(x, xc)[0, 1]) if xc.std() > 0 and x.std() > 0 else np.nan
            if len(rts2):
                k = int(np.argmin(np.abs(rts2 - apex_rt))); _, mz2, it2 = f2[k]; kp_ = mz2 < m + 1.5; frag = sorted(zip(np.round(mz2[kp_], 3), it2[kp_]), key=lambda kv: -kv[1])[:8]
                row["mse_top"] = ";".join(f"{a}:{b:.0f}" for a, b in frag); row["mse_rt_gap"] = float(abs(rts2[k] - apex_rt))
                for lab, fm in t_["frags"].items():
                    row[f"frag_{lab}"] = peak_in(mz2, it2, fm)
        out.append(row)
    return out

per_id = OUT / "identity_per_sample.csv"; P = mn.cached(per_id)
if P is None and HAVE_LUNG_RAW and mn.raw_available("lung*"):
    jobs = injections(); tg = targets(); t0 = time.time(); print(f"identity pass: {len(jobs)} injections x {len(tg)} bins...", flush=True)
    res = Parallel(n_jobs=6)(delayed(process_identity)((p, c, co, tg)) for p, c, co in jobs); P = pd.DataFrame([r_ for rows_ in res for r_ in rows_]); P.to_csv(per_id, index=False); print(f"{time.time() - t0:.0f} s")
HAVE_IDENTITY = P is not None

identity pass: 236 injections x 41 bins...


437 s


In [25]:
if HAVE_IDENTITY:
    rows = []
    for (b, name), g in P.groupby(["bin", "target"], sort=False):
        name = HYP.get(int(b), {}).get("name", name); det = g[g.detected]
        r_ = dict(bin=b, target=name, mz=g.mz.iloc[0], n=len(g), detected=int(g.detected.sum()), apex_rt_median=det.apex_rt.median(), apex_rt_sd=det.apex_rt.std(), I_apex_median=det.I_apex.median(), M1_over_M=det.M1_over_M.median(), M2_over_M=det.M2_over_M.median(), Mhalf_over_M=det.Mhalf_over_M.median())
        for c in [c for c in det.columns if c.startswith(("rel_", "corr_", "frag_"))]:
            v = det[c].astype(float); r_[c] = float(np.nanmedian(v)) if c.startswith(("rel_", "corr_")) else float((v > 0).mean())
        for cls in ("LC", "TB"):
            for co in ("discovery", "validation"):
                r_[f"I_{cls}_{co[:3]}"] = det[(det.cls == cls) & (det.cohort == co)].I_apex.median()
        dd = det[det.cohort == "discovery"]; r_["rt_dep_disc"] = dd.rt_dep.median(); r_["rt_dep_val"] = det[det.cohort == "validation"].rt_dep.median(); r_["apex_rt_f1_disc"] = dd.apex_rt.median()
        r_["apex_rt_f2"] = dd["apex_rt_f2"].median() if "apex_rt_f2" in dd else np.nan; r_["I_apex_f2"] = dd["I_apex_f2"].median() if "I_apex_f2" in dd else np.nan; r_["I_f1_at_dep"] = dd.I_f1_at_dep.median(); r_["I_f2_at_dep"] = dd["I_f2_at_dep"].median() if "I_f2_at_dep" in dd else np.nan
        f1_ok = abs(r_["apex_rt_f1_disc"] - r_["rt_dep_disc"]) <= 0.15; f2_ok = pd.notna(r_["apex_rt_f2"]) and abs(r_["apex_rt_f2"] - r_["rt_dep_disc"]) <= 0.15
        r_["max_function"] = "f1" if f1_ok and (not f2_ok or r_["I_f1_at_dep"] >= r_["I_f2_at_dep"]) else ("f2 (MS^E)" if f2_ok else "other"); r_["val_same_rt"] = abs(r_["rt_dep_val"] - r_["rt_dep_disc"]) <= 0.15
        top_dep = pd.Series([f for s in (det["mse_top_at_dep"].dropna() if "mse_top_at_dep" in det else []) for f in [x.split(":")[0] for x in s.split(";")]]).value_counts().head(8); r_["mse_frequent_at_dep"] = ";".join(f"{k}({v})" for k, v in top_dep.items())
        r_["ratio_LC_TB_disc"] = r_["I_LC_dis"] / r_["I_TB_dis"] if r_["I_TB_dis"] else np.nan; r_["ratio_LC_TB_val"] = r_["I_LC_val"] / r_["I_TB_val"] if r_["I_TB_val"] else np.nan
        top = pd.Series([f for s in det.mse_top.dropna() for f in [x.split(":")[0] for x in s.split(";")]]).value_counts().head(8); r_["mse_frequent_fragments"] = ";".join(f"{k}({v})" for k, v in top.items()); rows.append(r_)
    ID = pd.DataFrame(rows); ID.to_csv(OUT / "identity_summary.csv", index=False); IDn = ID.set_index("bin")
    display(ID[["bin", "target", "max_function", "detected", "apex_rt_f1_disc", "rt_dep_disc", "rt_dep_val", "I_apex_median", "M1_over_M", "Mhalf_over_M", "rel_Na", "rel_NH4", "ratio_LC_TB_disc", "ratio_LC_TB_val"]].round(3))
    r131 = IDn.loc[131]; r25 = IDn.loc[25]
else:
    print("identity table absent: Tables S16-S17 and the Table 4 check are skipped")

,bin,target,max_function,detected,apex_rt_f1_disc,rt_dep_disc,rt_dep_val,I_apex_median,M1_over_M,Mhalf_over_M,rel_Na,rel_NH4,ratio_LC_TB_disc,ratio_LC_TB_val
0,2341,unidentified doubly charged ion 932.144 (0.502 above 931.642),f2 (MS^E),236,11.202,11.199,11.193,1150.809,0.084,0.258,0.000,0.049,2.044,1.885
1,25,early-eluting ion 100.076 (C5H10NO+; not a valine/betaine fragment),f1,236,2.033,2.033,2.048,44700.250,0.058,0.000,0.122,0.000,0.371,0.308
2,184,unidentified ion 419.261 (M+1/M 0.09),f1,236,2.803,2.809,2.817,928.427,0.089,0.000,0.000,0.000,0.518,0.200
3,213,unidentified ion 477.303 (M+1/M 0.10),f1,236,2.987,2.987,3.991,681.204,0.098,0.000,0.000,0.000,0.598,0.293
4,1704,unidentified doubly charged ion 609.935,f1,236,8.174,8.174,8.181,1160.926,0.149,0.445,0.000,0.000,1.802,2.411
5,2340,unidentified doubly charged ion 931.642,f2 (MS^E),236,11.202,11.200,11.186,1945.968,0.155,0.380,0.000,0.056,1.563,1.569
6,131,phenylalanylphenylalanine [M+H]+,f1,236,2.333,2.333,2.347,24025.688,0.188,0.000,0.013,0.000,1.782,1.764
7,502,choline fragment 104.107 (LPC 16:0 in-source),f2 (MS^E),236,5.783,5.787,5.794,291.797,0.000,0.000,0.000,0.000,1.084,1.141
8,689,LPC 16:0 [M+H-H2O]+,f2 (MS^E),236,5.783,5.787,5.787,490.625,0.148,0.000,0.000,0.000,1.162,1.054
9,698,LPC 16:0 [M+H]+,f1,236,5.783,5.783,5.790,504550.250,0.275,0.000,0.017,0.000,1.288,1.192


### 7.4 Table 4 of the article: the curated rows

The identity labels and evidence sentences were curated by hand; the rank, retained m/z, retention time,
LC/TB ratios and role come from the ranking table above. The cell writes the LaTeX rows from which Table 4 of
the manuscript and Table S17 of the Supporting Information were assembled.

In [26]:
LPC16_FAMILY = [698, 689, 502, 945, 946, 542, 493, 699, 579, 512]
ROWS = [
    (2341, r"unidentified doubly charged ion, 0.502 above m/z 931.642 (both functions; stronger in the high-energy one)", r"isotope spacing 0.502 (0.26); XIC correlation with 931.642 $r = 0.68$"),
    (25, r"unidentified early-eluting ion, C$_5$H$_{10}$NO$^{+}$ (not a valine or betaine fragment)", r"M+1/M 0.06; $[\mathrm{M{+}Na}]^{+}$ 0.12; no 118.086 co-elution; 3$\times$ higher in TB"),
    (184, r"unidentified ion", r"M+1/M 0.09; another ion is the retained maximum of the bin in 52\% of validation samples"),
    (213, r"unidentified ion", r"M+1/M 0.10; another ion is the retained maximum of the bin in 97\% of validation samples"),
    (1704, r"unidentified doubly charged ion", r"isotope spacing 0.502 (0.45); $M \approx 1218$~Da"),
    (2340, r"unidentified doubly charged ion (both functions; stronger in the high-energy one)", r"isotope spacing 0.502 (0.38); $M \approx 1861$~Da"),
    (131, r"phenylalanylphenylalanine $[\mathrm{M{+}H}]^{+}$ (biomarker of the source study)", r"M+1/M 0.19; co-eluting $[\mathrm{M{+}Na}]^{+}$ ($r = 0.79$); MS$^{E}$ 120.081 (99\%), 103.054 (81\%), 166.086 (61\%); RT of the source study"),
    ("LPC16", r"lysoPC 16:0 $[\mathrm{M{+}H}]^{+}$ with its M+2 isotopologue (498.347), dimer (991.673, 992.676) and, in the high-energy function, its fragments 104.107, 86.097, 184.073, 258.110, 125.000 and water-loss ion 478.329", r"class fragments 184.073/104.107/86.096 at the apex in 100\% of injections; $[\mathrm{M{+}Na}]^{+}$ $r = 0.91$; dimer $r = 0.97$; isotopologue $r = 1.00$"),
    (1701, r"unidentified ion", r"M+1/M 0.24, M+2/M 0.12"),
    (662, r"linoleoylcarnitine (C18:2) $[\mathrm{M{+}H}]^{+}$", r"$\Delta m$ 0.3~ppm; M+1/M 0.23; MS$^{E}$ 85.028 (83\%), acylium 263.237 (27\%)"),
    (1735, r"unidentified doubly charged ion", r"isotope spacing 0.502 (0.29)"),
    (1059, r"retinol $[\mathrm{M{+}H{-}H_2O}]^{+}$ / anhydroretinol $[\mathrm{M{+}H}]^{+}$ (source-study annotation table)", r"co-eluting retinol $[\mathrm{M{+}H}]^{+}$ 287.237 ($r = 0.71$); MS$^{E}$ 119.086 (75\%), 95.086 (72\%), 157.101 (56\%)"),
    (35, r"phenylalanine immonium fragment of the dipeptide (in-source ion of the low-energy function on the discovery day; high-energy-function fragment on the validation day)", r"discovery day: low-energy 120.081 co-elutes with the dipeptide 313.155 ($r = 0.99$) and with 166.086 ($r = 0.95$); validation day: no low-energy 120.081 at 2.34~min (the low-energy ion at 2.0~min is the immonium of free phenylalanine, $r = 0.99$ with 166.086) and the retained maximum is the high-energy-function fragment; MS$^{E}$ 103.054 (91\%)"),
    (710, r"lysoPC 18:2 $[\mathrm{M{+}H}]^{+}$", r"$[\mathrm{M{+}Na}]^{+}$ $r = 0.90$; MS$^{E}$ 184.073/104.107 (100\%)"),
    (749, r"unidentified ion of the high-energy function (the polypropylene-glycol decamer of the same m/z elutes at 4.15~min in lung-cancer sera, is not detected in tuberculosis sera, and is not the bin maximum)", r"low-energy function at 5.79~min: 37 counts; high-energy function: $7.5 \times 10^{3}$ counts in both groups"),
    (711, r"lysoPC 18:1 $[\mathrm{M{+}H}]^{+}$", r"M+1/M 0.28; MS$^{E}$ 184.073/104.107 (100\%)"),
    (650, r"palmitoylcarnitine (C16) $[\mathrm{M{+}H}]^{+}$", r"$\Delta m$ 1~ppm; M+1/M 0.20; MS$^{E}$ 85.028 (86\%), 341.269 (44\%), acylium 239.237 (39\%)"),
    (663, r"oleoylcarnitine (C18:1) $[\mathrm{M{+}H}]^{+}$", r"M+1/M 0.23; MS$^{E}$ 85.028 (91\%), 367.284 (34\%), acylium 265.253 (35\%)"),
    (691, r"lysoPE 18:0 $[\mathrm{M{+}H}]^{+}$ (source-study annotation table)", r"$\Delta m$ 0.6~ppm; M+1/M 0.22; $[\mathrm{M{+}Na}]^{+}$ 0.06; MS$^{E}$ 341.305 (39\%); RT of the source study"),
    (734, r"lysoPC 22:6 $[\mathrm{M{+}H}]^{+}$", r"$[\mathrm{M{+}Na}]^{+}$ $r = 0.87$; MS$^{E}$ 184.073/104.107 (100\%)"),
]
T = T40.set_index("bin"); sel_set = set(T.index[T.selected]); lines_marked, lines_clean = [], []
for key, ident, evid in ROWS:
    if key == "LPC16":
        ranks_ = sorted(int(T.loc[b, "rank"]) for b in LPC16_FAMILY if b in T.index); rank = ", ".join(str(r_) for r_ in ranks_); mz = "496.3407"; rt = f"{T.loc[698, 'rt_median']:.2f}"
        r_d, r_v = f"{T.loc[698, 'ratio_LC_TB_disc']:.2f}", f"{T.loc[698, 'ratio_LC_TB_val']:.2f}"; role = "selected in some seeds" if any(b in sel_set for b in LPC16_FAMILY) else "ranked"
    else:
        r_ = T.loc[key]; rank = str(int(r_["rank"])); mz = f"{r_['mode_mz']:.4f}"; rt = f"{r_['rt_median']:.2f}"; r_d, r_v = f"{r_['ratio_LC_TB_disc']:.2f}", f"{r_['ratio_LC_TB_val']:.2f}"; role = "selected" if key in sel_set else "ranked"
    cells_ = [rank, mz, rt, ident, f"{r_d}; {r_v}", role]
    lines_marked.append(" & ".join("{\\color{blue}" + c + "}" for c in cells_) + r" \\"); lines_clean.append(" & ".join(cells_) + r" \\")
header = [r"\textbf{Rank}", r"\textbf{Retained m/z}", r"\textbf{RT}", r"\textbf{Retained ion}", r"\textbf{LC/TB disc.; val.}", r"\textbf{Role}"]
(TEX / "table4_rows_marked.tex").write_text(" & ".join("{\\color{blue}" + h + "}" for h in header) + " \\\\\n\\midrule\n" + "\n".join(lines_marked) + "\n", encoding="utf-8")
(TEX / "table4_rows_clean.tex").write_text(" & ".join(header) + " \\\\\n\\midrule\n" + "\n".join(lines_clean) + "\n", encoding="utf-8")
print("\n".join(lines_clean[:8]))


1 & 932.1439 & 11.19 & unidentified doubly charged ion, 0.502 above m/z 931.642 (both functions; stronger in the high-energy one) & 2.12; 1.80 & selected \\
2 & 100.0761 & 2.05 & unidentified early-eluting ion, C$_5$H$_{10}$NO$^{+}$ (not a valine or betaine fragment) & 0.36; 0.30 & selected \\
3 & 419.2610 & 2.81 & unidentified ion & 0.47; 0.22 & selected \\
4 & 477.3029 & 3.98 & unidentified ion & 0.55; 1.14 & selected \\
5 & 609.9349 & 8.18 & unidentified doubly charged ion & 1.80; 2.31 & ranked \\
6 & 931.6420 & 11.19 & unidentified doubly charged ion (both functions; stronger in the high-energy one) & 1.52; 1.46 & selected \\
7 & 313.1542 & 2.35 & phenylalanylphenylalanine $[\mathrm{M{+}H}]^{+}$ (biomarker of the source study) & 1.90; 1.67 & ranked \\
8, 9, 10, 14, 15, 16, 18, 23, 31, 33 & 496.3407 & 5.79 & lysoPC 16:0 $[\mathrm{M{+}H}]^{+}$ with its M+2 isotopologue (498.347), dimer (991.673, 992.676) and, in the high-energy function, its fragments 104.107, 86.097, 184.073, 258.11

### 7.5 Figure S4: co-eluting chromatograms (panels B and C; panel A comes from `milk.ipynb`)

In [27]:
LUNG_LC, LUNG_TB = "lcy20201008_102", "lcy20201008_62"; PPG_N = range(5, 14); trace_file = OUT / "xic_traces_lung.csv"
if mn.raw_available("lung*"):
    ppg_ions = {}
    for n_ in PPG_N:
        ppg_ions[f"PPG-{n_} [M+Na]+"] = n_ * PPG_UNIT + PPG_END + Na; ppg_ions[f"PPG-{n_} [M+NH4]+"] = n_ * PPG_UNIT + PPG_END + NH4
    parts = []
    for tag, name in (("C_lung_LC", LUNG_LC), ("D_lung_TB", LUNG_TB)):
        f = next(p for d_ in mn.RAW.glob("lung*") for p in d_.glob(f"{name}.mzML"))
        t_ = pd.DataFrame([dict(rt=rt, **{k: peak(mz, it, m, 15e-6) for k, m in ppg_ions.items()}) for rt, mz, it in read_scans(f, 2.4, 5.7, 1, function=1)])
        parts.append(t_.melt("rt", var_name="ion", value_name="I").assign(panel=tag))
    pd.concat(parts, ignore_index=True).to_csv(trace_file, index=False)
milk_traces = mn.RESULTS / "milk" / "xic_traces_milk.csv"
if trace_file.exists():
    dd = pd.read_csv(trace_file); has_milk = milk_traces.exists()
    if has_milk:
        dd = pd.concat([pd.read_csv(milk_traces), dd], ignore_index=True)
    fig = plt.figure(figsize=(7.6, 6.7 if has_milk else 3.6)); gs = fig.add_gridspec(2 if has_milk else 1, 2, height_ratios=[1, 1.05] if has_milk else [1], hspace=0.42, wspace=0.28, left=0.09, right=0.76, top=0.95, bottom=0.08)
    if has_milk:
        ax = fig.add_subplot(gs[0, :]); g = dd[dd.panel == "A_milk"]
        styles = {"[M+H]+ 222.097": ("#1f77b4", "-", 1.3), "[M+Na]+ 244.079": ("#17becf", "-", 1.1), "oxonium 204.087": ("#2ca02c", "-", 1.3), "186.076": ("#8c564b", "-", 1.1), "138.055 (``PABA'' ion)": ("#d62728", "-", 2.0), "126.055": ("#9467bd", "-", 1.1),
                  "PABA 120.044": ("0.45", ":", 1.6), "PABA [M+Na]+ 160.037": ("0.7", ":", 1.6), "Leu/Ile [M+H]+ 132.102": ("k", "--", 1.3)}
        ref_max = g[g.ion == "oxonium 204.087"].I.max()
        for ion, (c, ls_, lw) in styles.items():
            s = g[g.ion == ion]; top = s.I.max() if s.I.max() > 0 else 1; ax.plot(s.rt, s.I / (top if not ion.startswith("PABA") else ref_max), ls_, color=c, lw=lw, label=ion.replace("``", "\u201c").replace("''", "\u201d"))
        ax.set_xlim(5.9, 7.6); ax.set_ylim(0, 1.08); ax.set_xlabel("RT (min)"); ax.set_ylabel("intensity / apex of the ion"); ax.set_title("A  milk, conventional injection lai_c_01353 (batch 20220216): HexNAc ion family and the Leu/Ile competitor", fontsize=8.5, loc="left")
        ax.legend(fontsize=6.5, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.02)); ax.text(6.52, 1.03, "HexNAc (two anomers)", ha="center", fontsize=7); ax.text(7.21, 1.03, "Leu/Ile", ha="center", fontsize=7)
    cmap = plt.get_cmap("viridis"); ymax = dd[dd.panel.isin(["C_lung_LC", "D_lung_TB"])].I.max() * 1.05; row_ = 1 if has_milk else 0
    axC = fig.add_subplot(gs[row_, 0]); axD = fig.add_subplot(gs[row_, 1], sharey=axC)
    for ax, tag, title in ((axC, "C_lung_LC", f"B  lung-cancer serum {LUNG_LC}"), (axD, "D_lung_TB", f"C  tuberculosis serum {LUNG_TB} (same scale)")):
        g = dd[dd.panel == tag]
        for i, n_ in enumerate(PPG_N):
            c = cmap(i / (len(PPG_N) - 1)); s = g[g.ion == f"PPG-{n_} [M+Na]+"]; ax.plot(s.rt, s.I, color=c, lw=1.2, label=f"n = {n_}"); s = g[g.ion == f"PPG-{n_} [M+NH4]+"]; ax.plot(s.rt, s.I, "--", color=c, lw=0.9)
        ax.set_xlim(2.4, 5.7); ax.set_ylim(0, ymax); ax.set_xlabel("RT (min)"); ax.set_title(title, fontsize=8.5, loc="left")
    axC.set_ylabel("intensity (counts)"); plt.setp(axD.get_yticklabels(), visible=False); axD.legend(fontsize=6.5, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.02), title="PPG oligomer\n[M+Na]$^+$ solid\n[M+NH$_4$]$^+$ dashed", title_fontsize=6.5)
    for ax in fig.axes:
        ax.tick_params(labelsize=7.5)
    fig.savefig(FIG / "figS4_raw_check_xics.pdf", dpi=200); fig.savefig(FIG / "figS4_raw_check_xics.png", dpi=200); plt.close(fig); print("Figure S4 written" + ("" if has_milk else " (lung panels only: run milk.ipynb first for panel A)"))

Figure S4 written


## 8. The transform on one raw file

The deposited lung vectors were computed from all Waters functions (low energy, high energy and the
lock-mass reference), which is what `transform_file(path, grid, ms_levels=None)` does. The original run
stored the RT block in single precision, so differences of about 5·10⁻⁸ remain in that block.

In [28]:
if mn.raw_available("lung*"):
    for name in (LUNG_LC, LUNG_TB):
        f = next((p for d_ in mn.RAW.glob("lung*") for p in d_.glob(f"{name}.mzML")), None); row_ = runs[runs.sample_name == name]
        if f is None or row_.empty:
            continue
        v = mn.transform_file(f, GRID, ms_levels=None); dep = np.asarray(row_.output_vector.iloc[0], float)
        print(f"{name}: all scans -> max |difference| intensity {np.abs(v[:NB] - dep[:NB]).max():.1e}, m/z {np.abs(v[NB:2 * NB] - dep[NB:2 * NB]).max():.1e}, RT {np.abs(v[2 * NB:] - dep[2 * NB:]).max():.1e}")

lcy20201008_102: all scans -> max |difference| intensity 0.0e+00, m/z 0.0e+00, RT 0.0e+00


lcy20201008_62: all scans -> max |difference| intensity 0.0e+00, m/z 0.0e+00, RT 0.0e+00
